Attributablke

In [ ]:
from dask.distributed import Client, LocalCluster
from dask_jobqueue import SLURMCluster
import os
os.environ["PROJ_DATA"] = "/user/home/al18709/breathe/.pixi/envs/default/share/proj"
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import pickle
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Make sure notebook can find sbatch
os.environ['PATH'] += ':/system/steel-stack/slurm/23.11.6/bin'

cluster = SLURMCluster(
    queue="compute",
    account="GEOG022743",
    cores=6,
    processes=1,             # 6 workers per node
    memory="32GiB",         # per node
    walltime="4:00:00",
    # interface="eno1",        
)
cluster.scale(jobs=1)  
client = Client(cluster)
client

In [ ]:
population_df = pd.read_parquet('/bp1/geog-tropical/data/BREATHE/igr_population.parquet')

In [ ]:
def prepare_igr_geodataframe(results, geometry_path):
    """
    Merge IGR-level results with geometries for plotting.
    
    results      : the results dict from compute_summary
    geometry_path: path to shapefile/parquet with IGR geometries and 'rgi' column
    """
    geometry_gdf = gpd.read_file(geometry_path)
    geometry_gdf['rgi'] = geometry_gdf['rgi'].astype(str)

    # Pivot the leaf DataFrame from long (statistic as index) to wide (one column per statistic)
    igr_rows = []
    for igr, df in results['igr'].items():
        row = df['mean'].to_dict()
        row['p2.5']  = df['p2.5'].to_dict()
        row['p97.5'] = df['p97.5'].to_dict()
        row['rgi']   = str(igr)
        igr_rows.append(row)
    
    # Cleaner approach — stack into wide format directly
    igr_rows = []
    for igr, df in results['igr'].items():
        row = {'rgi': str(igr)}
        for statistic in df.index:
            row[f'{statistic}_mean']  = df.loc[statistic, 'mean']
            row[f'{statistic}_p2.5']  = df.loc[statistic, 'p2.5']
            row[f'{statistic}_p97.5'] = df.loc[statistic, 'p97.5']
        igr_rows.append(row)

    results_df = pd.DataFrame(igr_rows)
    results_gdf = geometry_gdf.merge(results_df, on='rgi', how='left')

    return results_gdf

In [ ]:
def annotate_capitals(ax, cities, fontsize=18):
    """Scatter and label Brazilian state capitals on a map axis."""
    for city in cities:
        marker = city.get('marker', 'o')
        s = city.get('s', 14)
        ax.scatter(city['x'], city['y'], color='black', s=s, marker=marker,zorder=1000)
        ax.annotate(city['name'], xy=(city['ax'], city['ay']), xycoords='data', fontsize=fontsize)


In [ ]:
def add_region_inset(fig, macro, region_color_map, position):
    """Add a small colour-coded region inset map to the figure."""
    ax_inset = fig.add_axes(position)
    for region in macro.index:
        color = region_color_map.get(region, 'lightgrey')
        macro.loc[[region]].plot(ax=ax_inset, color=color, edgecolor='white', alpha=0.4)
    ax_inset.axis('off')
    for region, row in macro.iterrows():
        centroid = row.geometry.centroid
        ax_inset.text(
            centroid.x, centroid.y, region,
            color=region_color_map.get(region, 'black'),
            fontsize=14, weight='bold', ha='center', va='center'
        )
    return ax_inset

In [ ]:
def plot_brazil(ax, fig, results_gdf, statistic='attributable_mortality_per_100k', change_levels=False):
    
    col = f'{statistic}_mean'

    cmap         = plt.get_cmap("plasma_r")
    cropped_cmap = cmap(np.linspace(0.1, 1.0, 256))
    new_cmap     = mcolors.ListedColormap(cropped_cmap)
    new_cmap.set_under('lightgrey')

    if change_levels:
        bounds = change_levels
        norm   = mcolors.BoundaryNorm(bounds, ncolors=256)
    elif 'per_100k' in statistic:
        bounds = [0, 20, 40, 60, 80, 100, 120, 140, 160, 180]
        norm   = mcolors.BoundaryNorm(bounds, ncolors=256, extend='max')
    elif 'fraction' in statistic:
        bounds = [0.00000000001, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7]
        norm   = mcolors.BoundaryNorm(bounds, ncolors=256, extend='max')
    else:
        bounds = [0, 20, 40, 60, 80, 100, 120, 140, 160, 180]
        norm   = mcolors.BoundaryNorm(bounds, ncolors=256, extend='max')

    results_gdf.plot(
        column=col, ax=ax, cmap=new_cmap, norm=norm,
        vmin=0, vmax=180, legend=False,
        edgecolor="#f5cca4", linewidth=0.2, alpha=0.9
    )

    sm      = plt.cm.ScalarMappable(cmap=new_cmap, norm=norm)
    sm._A   = []
    cbar_ax = fig.add_axes([0.15, 0.05, 0.35, 0.015])
    cbar    = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', pad=0, ticks=bounds, extend='both')

    if 'per_100k' in statistic:
        cbar.set_label("Attributable mortality (per 100,000) \n", fontsize=20, labelpad=10)
        ax.set_title("Heat-related mortality attributable to human-induced climate change (per 100,000)", fontsize=22, pad=16)
    elif 'heat_related' in statistic:
        cbar.set_label("Attributable mortality (percentage of factual heat-related deaths)", fontsize=20, labelpad=10)
        ax.set_title("Heat-related mortality attributable to human-induced climate change \n (percentage of heat-related deaths)", fontsize=22, pad=16)
    else:
        cbar.set_label("Attributable fraction (percentage of non-accidental deaths)", fontsize=20, labelpad=10)
        ax.set_title("Fraction of heat-related mortality attributable to human-induced climate change \n (percentage of non-accidental deaths)", fontsize=22, pad=16)

    cbar.outline.set_visible(False)
    cbar.ax.tick_params(labelsize=18)

    for spine in ["left", "right", "top", "bottom"]:
        ax.spines[spine].set_visible(False)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_xlabel("", fontsize=14)
    ax.set_ylabel("", fontsize=14)

    return ax, fig

In [ ]:
def load_results(path):
    with open(path, 'rb') as f:
        results = pickle.load(f)
    return results

In [ ]:
# ============================================================
# DATA LOADING
# ============================================================

# Load shapefiles
polys       = '/bp1/geog-tropical/data/BREATHE/RG2017_rgi_20180911/RG2017_rgi.shp'
gdf         = gpd.read_file(polys)
polys_states = '/bp1/geog-tropical/data/BREATHE/shapefiles/BR_UF_2022.shp'
gdf_states  = gpd.read_file(polys_states)

# Build macroregion polygons from state boundaries
macroregion_map = {
    'RO': 'North',    'AC': 'North',     'AM': 'North',       'RR': 'North',
    'PA': 'North',    'AP': 'North',     'TO': 'North',
    'MA': 'Northeast','PI': 'Northeast', 'CE': 'Northeast',   'RN': 'Northeast',
    'PB': 'Northeast','PE': 'Northeast', 'AL': 'Northeast',   'SE': 'Northeast', 'BA': 'Northeast',
    'MT': 'Central-West','MS': 'Central-West','GO': 'Central-West','DF': 'Central-West',
    'SP': 'Southeast','RJ': 'Southeast', 'MG': 'Southeast',   'ES': 'Southeast',
    'PR': 'South',    'SC': 'South',     'RS': 'South'
}
gdf_states["macroregion"] = gdf_states["SIGLA_UF"].map(macroregion_map)
macro = gdf_states.dissolve(by="macroregion")

# Load both result sets
results_100m = load_results('/bp1/geog-tropical/data/BREATHE/results/results_100M.pkl')
results_sim  = load_results('/bp1/geog-tropical/data/BREATHE/results/results_SIM.pkl')

# Merge both with geometries
results_gdf_100m = prepare_igr_geodataframe(results_100m, geometry_path=polys)
results_gdf_sim  = prepare_igr_geodataframe(results_sim,  geometry_path=polys)

# Derive region from rgi prefix
uf_code_to_region = {
    11: 'North', 12: 'North', 13: 'North', 14: 'North', 15: 'North', 16: 'North', 17: 'North',
    21: 'Northeast', 22: 'Northeast', 23: 'Northeast', 24: 'Northeast', 25: 'Northeast',
    26: 'Northeast', 27: 'Northeast', 28: 'Northeast', 29: 'Northeast',
    31: 'Southeast', 32: 'Southeast', 33: 'Southeast', 35: 'Southeast',
    41: 'South', 42: 'South', 43: 'South',
    50: 'Central-West', 51: 'Central-West', 52: 'Central-West', 53: 'Central-West',
}
for gdf_ in [results_gdf_100m, results_gdf_sim]:
    gdf_['Region'] = gdf_['rgi'].apply(
        lambda x: uf_code_to_region.get(int(str(x)[:2]), None)
    )

# Convenience alias (most plots use 100M)
results_gdf = results_gdf_100m

# Split into present/missing for plotting
df_merged   = results_gdf.dropna(subset=['attributable_mortality_per_100k_mean'])
gdf_missing = results_gdf[results_gdf['attributable_mortality_per_100k_mean'].isna()]

# RGI-to-region lookup for the 27 state capitals
rgi_region_map = {
    '120001': "North",        '270001': "Northeast",   '160001': "North",
    '130001': "North",        '290001': "Northeast",   '230001': "Northeast",
    '530001': "Central-West", '320001': "Southeast",   '520001': "Central-West",
    '210001': "Northeast",    '510001': "Central-West",'500001': "Central-West",
    '310001': "Southeast",    '150001': "North",       '250001': "Northeast",
    '410001': "South",        '260001': "Northeast",   '220001': "Northeast",
    '330001': "Southeast",    '240001': "Northeast",   '430001': "South",
    '110001': "North",        '140001': "North",       '420001': "South",
    '350001': "Southeast",    '280001': "Northeast",   '170001': "North"
}



In [ ]:
import math

def format_to_one_significant_figure(value):
    if pd.isna(value):
        return 'N/A'
    return f'{float(value):,.2f}'

def format_statistic_cell(results_dict, geographic_level, unit_name, statistic_name):
    dataframe = results_dict[geographic_level][unit_name]
    mean_value  = float(dataframe.loc[statistic_name, 'mean'])
    lower_bound = float(dataframe.loc[statistic_name, 'p2.5'])
    upper_bound = float(dataframe.loc[statistic_name, 'p97.5'])
    mean_formatted  = format_to_one_significant_figure(mean_value)
    lower_formatted = format_to_one_significant_figure(lower_bound)
    upper_formatted = format_to_one_significant_figure(upper_bound)
    return f"{mean_formatted} (95% CI: {lower_formatted}–{upper_formatted})"

all_statistics = list(results_100m['country']['Brazil'].index)

table_rows = [
    {
        'Statistic': statistic_name,
        '100M': format_statistic_cell(results_100m, 'country', 'Brazil', statistic_name),
        'SIM':  format_statistic_cell(results_sim,  'country', 'Brazil', statistic_name),
    }
    for statistic_name in all_statistics
]

display(pd.DataFrame(table_rows).set_index('Statistic'))

In [ ]:
# ============================================================
# HELPER: extract country/region scalar from results dict
# ============================================================

def get_stat(results, level, unit, statistic, col):
    return results[level][unit].loc[statistic, col]


# ============================================================
# PRINTED STATS
# ============================================================

for label, res in [('100M', results_100m), ('SIM', results_sim)]:
    factual_mean = get_stat(res, 'country', 'Brazil', 'factual_mortality',       'mean')
    factual_25   = get_stat(res, 'country', 'Brazil', 'factual_mortality',       'p2.5')
    factual_975  = get_stat(res, 'country', 'Brazil', 'factual_mortality',       'p97.5')
    attr_mean    = get_stat(res, 'country', 'Brazil', 'attributable_mortality',  'mean')
    attr_25      = get_stat(res, 'country', 'Brazil', 'attributable_mortality',  'p2.5')
    attr_975     = get_stat(res, 'country', 'Brazil', 'attributable_mortality',  'p97.5')
    af_mean      = get_stat(res, 'country', 'Brazil', 'heat_related_attributable_fraction', 'mean')
    af_25        = get_stat(res, 'country', 'Brazil', 'heat_related_attributable_fraction', 'p2.5')
    af_975       = get_stat(res, 'country', 'Brazil', 'heat_related_attributable_fraction', 'p97.5')

    print(f'\n{label}')
    print('Number of people who have died from heat-related causes between 2000-2018 in Brazil:')
    print(f'  {factual_mean:,.0f}  (per year: {factual_mean/19:,.0f})')
    print(f'  95% CI: {factual_25:,.0f} – {factual_mean:,.0f} – {factual_975:,.0f}')

    print('Fraction of heat-related deaths attributable to climate change:')
    print(f'  {af_25:.2f} – {af_mean:.2f} – {af_975:.2f} %')

    print('Number of attributable deaths between 2000-2018 in Brazil:')
    print(f'  {attr_mean:,.0f}  (per year: {attr_mean/19:,.0f})')
    print(f'  95% CI: {attr_25:,.0f} – {attr_mean:,.0f} – {attr_975:,.0f}')



In [ ]:
cities = [
    {'name': 'Rio Branco',     'x': -67.809, 'y':  -9.974, 'ax': -67.4,        'ay': -10.6},
    {'name': 'Maceió',         'x': -35.735, 'y':  -9.666, 'ax': -35.3,        'ay': -10.2},
    {'name': 'Macapá',         'x': -51.050, 'y':   0.034, 'ax': -50.6,        'ay':   0.0},
    {'name': 'Manaus',         'x': -60.021, 'y':  -3.122, 'ax': -59.6,        'ay':  -3.9},
    {'name': 'Salvador',       'x': -38.505, 'y': -12.986, 'ax': -38.2,        'ay': -13.6},
    {'name': 'Fortaleza',      'x': -38.529, 'y':  -3.724, 'ax': -38.2,        'ay':  -3.4},
    {'name': 'Brasília',       'x': -47.895, 'y': -15.807, 'ax': -49.9,        'ay': -15.3, 'marker': 's'},
    {'name': 'Vitória',        'x': -40.308, 'y': -20.315, 'ax': -39.9,        'ay': -21.0},
    {'name': 'Goiânia',        'x': -49.264, 'y': -16.686, 'ax': -48.9,        'ay': -17.4},
    {'name': 'São Luís',       'x': -44.302, 'y':  -2.530, 'ax': -44.0,        'ay':  -2.4},
    {'name': 'Cuiabá',         'x': -56.098, 'y': -15.601, 'ax': -55.7,        'ay': -16.1},
    {'name': 'Campo Grande',   'x': -54.620, 'y': -20.469, 'ax': -59.2,        'ay': -20.7},
    {'name': 'Belo Horizonte', 'x': -43.938, 'y': -19.920, 'ax': -43.5,        'ay': -19.8},
    {'name': 'Belém',          'x': -48.503, 'y':  -1.455, 'ax': -48.0,        'ay':  -2.1},
    {'name': 'João Pessoa',    'x': -34.864, 'y':  -7.121, 'ax': -34.4,        'ay':  -7.35},
    {'name': 'Curitiba',       'x': -49.265, 'y': -25.428, 'ax': -48.8,        'ay': -26.2},
    {'name': 'Recife',         'x': -34.883, 'y':  -8.059, 'ax': -34.5,        'ay':  -8.7},
    {'name': 'Teresina',       'x': -42.803, 'y':  -5.090, 'ax': -45.4,        'ay':  -5.8},
    {'name': 'Rio de Janeiro', 'x': -43.189, 'y': -22.917, 'ax': -42.8,        'ay': -23.7},
    {'name': 'Natal',          'x': -35.209, 'y':  -5.795, 'ax': -34.8,        'ay':  -6.0},
    {'name': 'Porto Alegre',   'x': -51.211, 'y': -30.038, 'ax': -50.8,        'ay': -30.8},
    {'name': 'Porto Velho',    'x': -63.903, 'y':  -8.761, 'ax': -63.5,        'ay':  -9.4},
    {'name': 'Boa Vista',      'x': -60.673, 'y':   2.823, 'ax': -60.2,        'ay':   2.9},
    {'name': 'Florianópolis',  'x': -48.548, 'y': -27.595, 'ax': -48.1,        'ay': -28.3},
    {'name': 'São Paulo',      'x': -46.638, 'y': -23.568, 'ax': -46.3,        'ay': -24.6},
    {'name': 'Aracaju',        'x': -37.073, 'y': -10.947, 'ax': -36.7,        'ay': -11.6},
    {'name': 'Palmas',         'x': -48.333, 'y': -10.184, 'ax': -47.9,        'ay': -10.9},
]

In [ ]:
import numpy as np

for igr, df in results_100m['igr'].items():
    # Select only numeric columns
    numeric_df = df.select_dtypes(include=[np.number])
    if np.isinf(numeric_df.values).any():
        print(f"IGR {igr} contains inf or -inf values")

In [ ]:
# ============================================================
# FIGURE: 100M vs SIM comparison (region-level CI chart + map)
# ============================================================

region_color_map = {
    'Brazil':       '#444444',
    'Central-West': '#1f77b4',
    'North':        '#ff7f0e',
    'Northeast':    '#2ca02c',
    'South':        '#d62728',
    'Southeast':    '#9467bd',
}

def _af(res, unit):
    s = res['region'][unit] if unit != 'Brazil' else res['country']['Brazil']
    return (
        s.loc['heat_related_attributable_fraction', 'mean'],
        s.loc['heat_related_attributable_fraction', 'p2.5'],
        s.loc['heat_related_attributable_fraction', 'p97.5'],
    )

chart_data = []
for region in ['South', 'Southeast', 'Central-West', 'Northeast', 'North', 'Brazil']:
    m100, lo100, hi100 = _af(results_100m, region)
    msim, losim, hisim = _af(results_sim,  region)
    chart_data.append((
        region,
        m100, lo100, hi100,
        msim, losim, hisim,
    ))


fig = plt.figure(figsize=(28, 18))
fig.suptitle("Low-income (100M) \u2212 whole population (SIM)", 
             fontsize=24, fontweight='bold', y=0.95)  # y>1 pushes it above the panels
gs  = gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[1, 3], wspace=0.15)


# ── Left: CI dot plot (panel a) ────────────────────────────────────────────
ax_plot = fig.add_subplot(gs[0, 0])
ax_plot.text(-0.28, 1.02, 'a.', transform=ax_plot.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

offset      = 0.15
row_spacing = 0.9
ytick_pos, ytick_labels = [], []

for i, (label, m100, lo100, hi100, msim, losim, hisim) in enumerate(chart_data):
    color = region_color_map[label]
    y     = i * row_spacing - 0.4

    ax_plot.plot([lo100, hi100], [y + offset, y + offset], color=color, lw=2.5, solid_capstyle='round')
    ax_plot.plot(m100, y + offset, 'o', color=color, ms=9, zorder=5)

    ax_plot.plot([losim, hisim], [y - offset, y - offset], color=color, lw=2.5, ls='--',
                 solid_capstyle='round', alpha=0.75)
    ax_plot.plot(msim, y - offset, 'o', color=color, ms=9, mfc='white', mew=2.5, zorder=5)

    ytick_pos.append(y)
    ytick_labels.append(label)

ax_plot.set_ylim(-1, (len(chart_data) - 1) * row_spacing + 1)
ax_plot.set_yticks(ytick_pos)
ax_plot.set_yticklabels(ytick_labels, fontsize=18)
ax_plot.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}'))
ax_plot.tick_params(axis='x', labelsize=16)
ax_plot.set_xlabel("Percentage of heat-deaths attributable \nto climate change (%)", fontsize=20, labelpad=10)
# ax_plot.set_title("National and regional summary", fontsize=24, fontweight='bold', pad=10)
ax_plot.grid(axis='x', linestyle='--', alpha=0.3)
for spine in ["left", "right", "top"]:
    ax_plot.spines[spine].set_visible(False)
ax_plot.yaxis.set_ticks_position('none')
ax_plot.tick_params(axis='y', length=0)
ax_plot.set_aspect('auto')

from matplotlib.lines import Line2D
ax_plot.legend(
    handles=[
        Line2D([0], [0], color='#444444', lw=2.5, marker='o', ms=8, label='100M'),
        Line2D([0], [0], color='#444444', lw=2.5, ls='--', marker='o', ms=8,
               markerfacecolor='white', markeredgewidth=2, label='SIM'),
    ],
    fontsize=16, loc='lower right', framealpha=0.7
)

# ── Right: difference map (panel b) ────────────────────────────────────────
results_gdf_100m['sim_100m_diff'] = (
    results_gdf_100m['heat_related_attributable_fraction_mean']
    - results_gdf_sim['heat_related_attributable_fraction_mean']
)

ax_map = fig.add_subplot(gs[0, 1])
ax_map.text(0.05, 1.02, 'b.', transform=ax_map.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

levels = [-5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5]

results_gdf_100m.plot(
    column='sim_100m_diff', ax=ax_map,
    cmap='coolwarm', vmin=-6, vmax=6, legend=False,
    norm=mcolors.BoundaryNorm(levels, ncolors=256, extend='both'),
    edgecolor="#f0f2f3", linewidth=0.2, alpha=0.9
)
sm    = plt.cm.ScalarMappable(cmap='coolwarm',
          norm=mcolors.BoundaryNorm(levels, ncolors=256, extend='both'))
sm._A = []

gdf_missing.plot(ax=ax_map, color='none', edgecolor="#c1c3c5", hatch='////', linewidth=0.1)
macro.plot(ax=ax_map, edgecolor="#ffffff", linewidth=1.5, facecolor="none")
ax_map.set_aspect('auto')
# ax_map.set_title("Low-income (100M) - Whole population (SIM)", fontsize=24, fontweight='bold', pad=16)
for spine in ["left", "right", "top", "bottom"]:
    ax_map.spines[spine].set_visible(False)
ax_map.set_xticks([]); ax_map.set_yticks([])
annotate_capitals(ax_map, cities)

# Draw colorbar after tight_layout so positions are finalised
# plt.tight_layout()
plt.tight_layout(rect=[0, 0, 1, 0.97])  # leaves room at top for suptitle
fig.canvas.draw()

pos = ax_map.get_position()
cbar_ax = fig.add_axes([
    pos.x0 + pos.width * 0.1,
    pos.y0 - 0.06,
    pos.width * 0.8,
    0.015
])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', ticks=levels, extend='both')
cbar.set_label("Difference in percentage of heat-deaths attributable to climate change (%)",
               fontsize=18, labelpad=10)
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=16)

# Inset region map — sits over top-left of the map panel
pos_plot = ax_plot.get_position()
ax_inset = fig.add_axes([pos_plot.x0 + 0.1, pos_plot.y1 - 0.28, 0.15, 0.30])
for region in macro.index:
    macro.loc[[region]].plot(ax=ax_inset, color=region_color_map.get(region, 'lightgrey'),
                             edgecolor='white', alpha=0.4)
    centroid = macro.loc[region, 'geometry'].centroid
    ax_inset.text(centroid.x, centroid.y, region,
                  color=region_color_map.get(region, 'black'),
                  fontsize=14, weight='bold', ha='center', va='center')
ax_inset.axis('off')

plt.savefig('Figure3_heat-related_fraction.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
from scipy.stats import kstwobign

def weighted_ks_test(x, y, weights_x, weights_y, thresh=0.05):
    """
    Two-sample Kolmogorov-Smirnov test, weighted version.
    Assumes weights_x and weights_y are already normalized (sum to 1).
    """

    # --- Effective sample size ---
    # If some data points have huge weights and others tiny weights, the 113,000 samples
    # don't really behave like 113,000 independent
    # observations, a few heavily-weighted points dominate. The
    # "effective sample size" (Kish's formula) estimates how many
    # equally-weighted points would give you the same statistical power.
    effective_n_x = 1 / np.sum(weights_x ** 2)
    effective_n_y = 1 / np.sum(weights_y ** 2)

    # Combine into one effective sample size for the pair, the same way
    # the standard KS test combines n and m: (n*m)/(n+m)
    combined_effective_n = (effective_n_x * effective_n_y) / (effective_n_x + effective_n_y)
    # print('this is probably 1:',combined_effective_n)

    def weighted_cdf(eval_points, data, weights):
        """
        Weighted empirical CDF: for each value in eval_points, returns the
        (weighted) fraction of `data` that is <= that value.
        """
        order = np.argsort(data)
        data_sorted = data[order]
        weights_sorted = weights[order]
        cumulative_weight = np.cumsum(weights_sorted)

        # Find where each eval point slots into the sorted data -- i.e.
        # how much cumulative weight lies at or below it.
        insert_positions = np.searchsorted(data_sorted, eval_points, side='right') - 1

        # insert_position == -1 means the eval point is smaller than
        # everything in `data`, so its CDF value is 0.
        return np.where(insert_positions >= 0, cumulative_weight[insert_positions], 0.0)

    # The KS statistic is the largest gap between the two CDFs. CDFs only
    # change at data points, so we only need to check the gap at every x
    # and y value -- that's why we concatenate them: it's just the list of
    # candidate points to check, not merging the two distributions.
    candidate_points = np.concatenate([x, y])
    cdf_x_at_points = weighted_cdf(candidate_points, x, weights_x)
    cdf_y_at_points = weighted_cdf(candidate_points, y, weights_y)

    biggest_gap = np.max(np.abs(cdf_x_at_points - cdf_y_at_points))

    # `thresh` is an effect-size floor: gaps smaller than `thresh` are
    # treated as "not meaningfully different" and clipped to 0.
    D_statistic = max(biggest_gap - thresh, 0)

    if D_statistic == 0:
        return {"statistic": 0.0, "p_value": 1.0, "n_eff": combined_effective_n}

    # Standard asymptotic KS p-value formula, using effective n instead of
    # plain n -- same limiting distribution scipy's own ks_2samp uses.
    p_value = kstwobign.sf(np.sqrt(combined_effective_n) * D_statistic)
    p_value = min(1.0, max(2.2e-16, p_value))

    return {"statistic": D_statistic, "p_value": p_value, "n_eff": combined_effective_n}

In [ ]:
# open raw results pickle file to extract the raw data for the KS test
with open("/bp1/geog-tropical/data/BREATHE/results/aggregated_SIM.pkl", "rb") as f:
    results_sim = pickle.load(f)

with open("/bp1/geog-tropical/data/BREATHE/results/aggregated_100M.pkl", "rb") as f:
    results_100m = pickle.load(f)

weights = np.load('/bp1/geog-tropical/data/BREATHE/results/aggregated_100M_weights.npy')

with open("/bp1/geog-tropical/data/BREATHE/results/aggregated_100M_region.pkl", "rb") as f:
    results_100m_region = pickle.load(f)
    
with open("/bp1/geog-tropical/data/BREATHE/results/aggregated_SIM_region.pkl", "rb") as f:
    results_sim_region = pickle.load(f)


In [ ]:
attributable_sim = np.array(results_sim['Brazil'].factual_mortality - results_sim['Brazil'].counterfactual_mortality)
attributable_100m = np.array(results_100m['Brazil'].factual_mortality - results_100m['Brazil'].counterfactual_mortality)

for r in ['North', 'Northeast', 'Central-West', 'Southeast', 'South']:
    attributable_sim_region = np.array(results_sim_region[r].factual_mortality - results_sim_region[r].counterfactual_mortality)
    attributable_100m_region = np.array(results_100m_region[r].factual_mortality - results_100m_region[r].counterfactual_mortality)
    result = weighted_ks_test(attributable_sim_region, attributable_100m_region, weights, weights, thresh=0.05)
    print(f"Region: {r}, KS statistic: {result['statistic']:.4f}, p-value: {result['p_value']:.4f}, effective n: {result['n_eff']:.2f}")
    

result = weighted_ks_test(attributable_sim, attributable_100m, weights, weights, thresh=0.05)

In [ ]:
def compute_weighted_cdf(data, weights):
    weights = weights / np.sum(weights)
    order = np.argsort(data)
    sorted_data_values = data[order]      # x-axis: the actual data, sorted
    sorted_weights = weights[order]        # weights reordered to match
    cumulative_probability = np.cumsum(sorted_weights)  # y-axis: CDF values
    return sorted_data_values, cumulative_probability

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Convert to thousands for readability
sim_thousands = attributable_sim / 1000
m100_thousands = attributable_100m / 1000

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Unweighted histograms ---
axes[0, 0].hist(sim_thousands, bins=50, alpha=0.5, label='Factual (SIM)', color='C0')
axes[0, 0].hist(m100_thousands, bins=50, alpha=0.5, label='Counterfactual (100M)', color='C1')
axes[0, 0].set_title('Unweighted histogram')
axes[0, 0].set_xlabel('Attributable mortality (thousands of people)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()

# --- Weighted histograms ---
axes[0, 1].hist(sim_thousands, bins=50, weights=weights, alpha=0.5, label='Factual (SIM)', color='C0')
axes[0, 1].hist(m100_thousands, bins=50, weights=weights, alpha=0.5, label='Counterfactual (100M)', color='C1')
axes[0, 1].set_title('Weighted histogram')
axes[0, 1].set_xlabel('Attributable mortality (thousands of people)')
axes[0, 1].set_ylabel('Weighted count')
axes[0, 1].legend()

# --- Unweighted CDFs ---
x_sorted_unw = np.sort(sim_thousands)
y_sorted_unw = np.sort(m100_thousands)
axes[1, 0].step(x_sorted_unw, np.arange(1, len(x_sorted_unw)+1) / len(x_sorted_unw), 
                where='post', label='Factual (SIM)')
axes[1, 0].step(y_sorted_unw, np.arange(1, len(y_sorted_unw)+1) / len(y_sorted_unw), 
                where='post', label='Counterfactual (100M)')
axes[1, 0].set_title('Unweighted CDF')
axes[1, 0].set_xlabel('Attributable mortality (thousands of people)')
axes[1, 0].set_ylabel('Cumulative probability')
axes[1, 0].legend()

# --- Weighted CDFs ---
x_sorted, x_cdf = compute_weighted_cdf(sim_thousands, weights)
y_sorted, y_cdf = compute_weighted_cdf(m100_thousands, weights)
axes[1, 1].step(x_sorted, x_cdf, where='post', label='Factual (SIM)')
axes[1, 1].step(y_sorted, y_cdf, where='post', label='Counterfactual (100M)')
axes[1, 1].set_title('Weighted CDF')
axes[1, 1].set_xlabel('Attributable mortality (thousands of people)')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('histograms_and_cdfs_comparison.png', dpi=150)
plt.show()

print("Factual (SIM):        min=%.2f  max=%.2f  mean=%.2f (thousands)" % 
      (sim_thousands.min(), sim_thousands.max(), sim_thousands.mean()))
print("Counterfactual (100M): min=%.2f  max=%.2f  mean=%.2f (thousands)" % 
      (m100_thousands.min(), m100_thousands.max(), m100_thousands.mean()))

In [ ]:
attributable_sim = np.array((results_sim['Brazil'].factual_mortality - results_sim['Brazil'].counterfactual_mortality)/results_sim['Brazil'].factual_mortality)
attributable_100m = np.array((results_100m['Brazil'].factual_mortality - results_100m['Brazil'].counterfactual_mortality)/results_100m['Brazil'].factual_mortality)
result = weighted_ks_test(attributable_sim, attributable_100m, weights, weights, thresh=0.05)

In [ ]:
for r in ['North', 'Northeast', 'Central-West', 'Southeast', 'South']:
    attributable_sim_region = np.array((results_sim_region[r].factual_mortality - results_sim_region[r].counterfactual_mortality) / results_sim_region[r].factual_mortality)
    attributable_100m_region = np.array((results_100m_region[r].factual_mortality - results_100m_region[r].counterfactual_mortality) / results_100m_region[r].factual_mortality)
    result = weighted_ks_test(attributable_sim_region, attributable_100m_region, weights, weights, thresh=0.05)
    print(f"Region: {r}, KS statistic: {result['statistic']}, p-value: {result['p_value']}, effective n: {result['n_eff']}")
    

In [ ]:
# Convert to thousands for readability
sim_thousands = attributable_sim * 100
m100_thousands = attributable_100m * 100

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Unweighted histograms ---
axes[0, 0].hist(sim_thousands, bins=50, alpha=0.5, label='SIM', color='C0')
axes[0, 0].hist(m100_thousands, bins=50, alpha=0.5, label='100M', color='C1')
axes[0, 0].set_title('Unweighted histogram')
axes[0, 0].set_xlabel('Attributable mortality (thousands of people)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()

# --- Weighted histograms ---
axes[0, 1].hist(sim_thousands, bins=50, weights=weights, alpha=0.5, label='SIM', color='C0')
axes[0, 1].hist(m100_thousands, bins=50, weights=weights, alpha=0.5, label='100M', color='C1')
axes[0, 1].set_title('Weighted histogram')
axes[0, 1].set_xlabel('Attributable mortality (thousands of people)')
axes[0, 1].set_ylabel('Weighted count')
axes[0, 1].legend()

# --- Unweighted CDFs ---
x_sorted_unw = np.sort(sim_thousands)
y_sorted_unw = np.sort(m100_thousands)
axes[1, 0].step(x_sorted_unw, np.arange(1, len(x_sorted_unw)+1) / len(x_sorted_unw), 
                where='post', label='Factual (SIM)')
axes[1, 0].step(y_sorted_unw, np.arange(1, len(y_sorted_unw)+1) / len(y_sorted_unw), 
                where='post', label='Counterfactual (100M)')
axes[1, 0].set_title('Unweighted CDF')
axes[1, 0].set_xlabel('Attributable mortality (thousands of people)')
axes[1, 0].set_ylabel('Cumulative probability')
axes[1, 0].legend()

# --- Weighted CDFs ---
x_sorted, x_cdf = compute_weighted_cdf(sim_thousands, weights)
y_sorted, y_cdf = compute_weighted_cdf(m100_thousands, weights)
axes[1, 1].step(x_sorted, x_cdf, where='post', label='Factual (SIM)')
axes[1, 1].step(y_sorted, y_cdf, where='post', label='Counterfactual (100M)')
axes[1, 1].set_title('Weighted CDF')
axes[1, 1].set_xlabel('percentage of heat-deaths attributable to climate change')
axes[1, 1].set_ylabel('Cumulative probability')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('histograms_and_cdfs_comparison.png', dpi=150)
plt.show()

print("SIM:        min=%.2f  max=%.2f  mean=%.2f (thousands)" % 
      (sim_thousands.min(), sim_thousands.max(), sim_thousands.mean()))
print("100M: min=%.2f  max=%.2f  mean=%.2f (thousands)" % 
      (m100_thousands.min(), m100_thousands.max(), m100_thousands.mean()))

In [ ]:
# print summary statistics

# Brazil-level results
# 100m
print('==========\n Brazil Statistics\n==========')
print('100m brazil 2.5th percentile', results_100m['country']['Brazil'].loc['heat_related_attributable_fraction']['p2.5'])
print('100m Brazil mean',results_100m['country']['Brazil'].loc['heat_related_attributable_fraction']['mean'])
print('100m Brazil 97.5th percentile', results_100m['country']['Brazil'].loc['heat_related_attributable_fraction']['p97.5'])
print('\n')
# sim
print('sim brazil 2.5th percentile', results_sim['country']['Brazil'].loc['heat_related_attributable_fraction']['p2.5'])
print('sim Brazil mean',results_sim['country']['Brazil'].loc['heat_related_attributable_fraction']['mean'])
print('sim Brazil 97.5th percentile', results_sim['country']['Brazil'].loc['heat_related_attributable_fraction']['p97.5'])
print('\n\n')

print('==========\n north Statistics\n==========')
print('100m north 2.5th percentile', results_100m['region']['North'].loc['heat_related_attributable_fraction']['p2.5'])
print('100m North mean',results_100m['region']['North'].loc['heat_related_attributable_fraction']['mean'])
print('100m North 97.5th percentile', results_100m['region']['North'].loc['heat_related_attributable_fraction']['p97.5'])
print('\n')
# sim
print('sim North 2.5th percentile', results_sim['region']['North'].loc['heat_related_attributable_fraction']['p2.5'])
print('sim North mean',results_sim['region']['North'].loc['heat_related_attributable_fraction']['mean'])
print('sim North 97.5th percentile', results_sim['region']['North'].loc['heat_related_attributable_fraction']['p97.5'])
print('\n\n')

print('==========\n Northeast Statistics\n==========')
print('100m Northeast 2.5th percentile', results_100m['region']['Northeast'].loc['heat_related_attributable_fraction']['p2.5'])
print('100m Northeast mean',results_100m['region']['Northeast'].loc['heat_related_attributable_fraction']['mean'])
print('100m Northeast 97.5th percentile', results_100m['region']['Northeast'].loc['heat_related_attributable_fraction']['p97.5'])
print('\n')
# sim
print('sim Northeast 2.5th percentile', results_sim['region']['Northeast'].loc['heat_related_attributable_fraction']['p2.5'])
print('sim Northeast mean',results_sim['region']['Northeast'].loc['heat_related_attributable_fraction']['mean'])
print('sim Northeast 97.5th percentile', results_sim['region']['Northeast'].loc['heat_related_attributable_fraction']['p97.5'])
print('\n\n')

print('==========\n Central-West Statistics\n==========')
print('100m Central-West 2.5th percentile', results_100m['region']['Central-West'].loc['heat_related_attributable_fraction']['p2.5'])
print('100m Central-West mean',results_100m['region']['Central-West'].loc['heat_related_attributable_fraction']['mean'])
print('100m Central-West 97.5th percentile', results_100m['region']['Central-West'].loc['heat_related_attributable_fraction']['p97.5'])
print('\n')
# sim
print('sim Central-West 2.5th percentile', results_sim['region']['Central-West'].loc['heat_related_attributable_fraction']['p2.5'])
print('sim Central-West mean',results_sim['region']['Central-West'].loc['heat_related_attributable_fraction']['mean'])
print('sim Central-West 97.5th percentile', results_sim['region']['Central-West'].loc['heat_related_attributable_fraction']['p97.5'])
print('\n\n')


print('==========\n Southeast Statistics\n==========')
print('100m Southeast 2.5th percentile', results_100m['region']['Southeast'].loc['heat_related_attributable_fraction']['p2.5'])
print('100m Southeast mean',results_100m['region']['Southeast'].loc['heat_related_attributable_fraction']['mean'])
print('100m Southeast 97.5th percentile', results_100m['region']['Southeast'].loc['heat_related_attributable_fraction']['p97.5'])
print('\n')
# sim
print('sim Southeast 2.5th percentile', results_sim['region']['Southeast'].loc['heat_related_attributable_fraction']['p2.5'])
print('sim Southeast mean',results_sim['region']['Southeast'].loc['heat_related_attributable_fraction']['mean'])
print('sim Southeast 97.5th percentile', results_sim['region']['Southeast'].loc['heat_related_attributable_fraction']['p97.5'])
print('\n\n')

print('==========\n South Statistics\n==========')
print('100m South 2.5th percentile', results_100m['region']['South'].loc['heat_related_attributable_fraction']['p2.5'])
print('100m South mean',results_100m['region']['South'].loc['heat_related_attributable_fraction']['mean'])
print('100m South 97.5th percentile', results_100m['region']['South'].loc['heat_related_attributable_fraction']['p97.5'])
print('\n')
# sim
print('sim South 2.5th percentile', results_sim['region']['South'].loc['heat_related_attributable_fraction']['p2.5'])
print('sim South mean',results_sim['region']['South'].loc['heat_related_attributable_fraction']['mean'])
print('sim South 97.5th percentile', results_sim['region']['South'].loc['heat_related_attributable_fraction']['p97.5'])
print('\n\n')

# ['heat_related_attributable_fraction', 'mean']

# for region in ['South', 'Southeast', 'Central-West', 'Northeast', 'North', 'Brazil']:
#     m100, lo100, hi100 = _af(results_100m, region)
#     msim, losim, hisim = _af(results_sim,  region)
#     chart_data.append((
#         region,
#         m100, lo100, hi100,
#         msim, losim, hisim,
#     ))

In [ ]:
fig = plt.figure(figsize=(40, 18))
gs  = gridspec.GridSpec(nrows=1, ncols=2, wspace=0.02, bottom=0.12)

levels = [0,10,20,30,40,50,60,70,80,90,100]
norm   = mcolors.BoundaryNorm(levels, ncolors=256, extend='both')
sm     = plt.cm.ScalarMappable(cmap='coolwarm', norm=norm)
sm._A  = []

# Left: 100M
ax_100m = fig.add_subplot(gs[0, 0])
ax_100m.text(0.05, 1.02, 'a.', transform=ax_100m.transAxes, fontsize=28, fontweight='bold', va='bottom')
results_gdf_100m.plot(
    column='heat_related_attributable_fraction_mean', ax=ax_100m,
    cmap='coolwarm', norm=norm, legend=False,
    edgecolor="#f0f2f3", linewidth=0.2, alpha=0.9
)
gdf_missing.plot(ax=ax_100m, color='none', edgecolor="#c1c3c5", hatch='////', linewidth=0.1)
macro.plot(ax=ax_100m, edgecolor="#ffffff", linewidth=1.5, facecolor="none")
ax_100m.set_title("100M", fontsize=26, fontweight='bold', pad=16)
for spine in ax_100m.spines.values(): spine.set_visible(False)
ax_100m.set_xticks([]); ax_100m.set_yticks([])
ax_100m.set_aspect('auto')
annotate_capitals(ax_100m, cities)

# Right: SIM
ax_sim = fig.add_subplot(gs[0, 1])
ax_sim.text(0.05, 1.02, 'b.', transform=ax_sim.transAxes, fontsize=28, fontweight='bold', va='bottom')
results_gdf_sim.plot(
    column='heat_related_attributable_fraction_mean', ax=ax_sim,
    cmap='coolwarm', norm=norm, legend=False,
    edgecolor="#f0f2f3", linewidth=0.2, alpha=0.9
)
gdf_missing.plot(ax=ax_sim, color='none', edgecolor="#c1c3c5", hatch='////', linewidth=0.1)
macro.plot(ax=ax_sim, edgecolor="#ffffff", linewidth=1.5, facecolor="none")
ax_sim.set_title("SIM", fontsize=26, fontweight='bold', pad=16)
for spine in ax_sim.spines.values(): spine.set_visible(False)
ax_sim.set_xticks([]); ax_sim.set_yticks([])
ax_sim.set_aspect('auto')
annotate_capitals(ax_sim, cities)

# Shared horizontal colorbar centred below both panels
cbar_ax = fig.add_axes([0.25, 0.04, 0.5, 0.015])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', ticks=levels, extend='both')
cbar.set_label("Climate change attributable heat-mortality as a fraction of factual heat-related mortality (%)", fontsize=22, labelpad=10)
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=18)

plt.savefig('figure_4.svg', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
levels = [0,10,20,30,40,50,60,70,80,90,100]
norm   = mcolors.BoundaryNorm(levels, ncolors=256, extend='both')
sm     = plt.cm.ScalarMappable(cmap='coolwarm', norm=norm)
sm._A  = []

In [ ]:
fig_a = plt.figure(figsize=(20, 18))
ax_100m = fig_a.add_subplot(111)
ax_100m.text(0.05, 1.02, 'a.', transform=ax_100m.transAxes, fontsize=28, fontweight='bold', va='bottom')

results_gdf_100m.plot(
    column='heat_related_attributable_fraction_mean', ax=ax_100m,
    cmap='coolwarm', norm=norm, legend=False,
    edgecolor="#f0f2f3", linewidth=0.2, alpha=0.9
)
gdf_missing.plot(ax=ax_100m, color='none', edgecolor="#c1c3c5", hatch='////', linewidth=0.1)
macro.plot(ax=ax_100m, edgecolor="#ffffff", linewidth=1.5, facecolor="none")
ax_100m.set_title("100M", fontsize=26, fontweight='bold', pad=16)
for spine in ax_100m.spines.values(): spine.set_visible(False)
ax_100m.set_xticks([]); ax_100m.set_yticks([])
ax_100m.set_aspect('auto')
annotate_capitals(ax_100m, cities)

cbar_ax = fig_a.add_axes([0.25, 0.04, 0.5, 0.015])
cbar = fig_a.colorbar(sm, cax=cbar_ax, orientation='horizontal', ticks=levels, extend='both')
cbar.set_label("Climate change attributable heat-mortality as a fraction of factual heat-related mortality (%)", fontsize=22, labelpad=10)
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=18)

fig_a.savefig('figure_4a.svg', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
fig_b = plt.figure(figsize=(20, 18))
ax_sim = fig_b.add_subplot(111)
ax_sim.text(0.05, 1.02, 'b.', transform=ax_sim.transAxes, fontsize=28, fontweight='bold', va='bottom')

results_gdf_sim.plot(
    column='heat_related_attributable_fraction_mean', ax=ax_sim,
    cmap='coolwarm', norm=norm, legend=False,
    edgecolor="#f0f2f3", linewidth=0.2, alpha=0.9
)
gdf_missing.plot(ax=ax_sim, color='none', edgecolor="#c1c3c5", hatch='////', linewidth=0.1)
macro.plot(ax=ax_sim, edgecolor="#ffffff", linewidth=1.5, facecolor="none")
ax_sim.set_title("SIM", fontsize=26, fontweight='bold', pad=16)
for spine in ax_sim.spines.values(): spine.set_visible(False)
ax_sim.set_xticks([]); ax_sim.set_yticks([])
ax_sim.set_aspect('auto')
annotate_capitals(ax_sim, cities)

cbar_ax = fig_b.add_axes([0.25, 0.04, 0.5, 0.015])
cbar = fig_b.colorbar(sm, cax=cbar_ax, orientation='horizontal', ticks=levels, extend='both')
cbar.set_label("Climate change attributable heat-mortality as a fraction of factual heat-related mortality (%)", fontsize=22, labelpad=10)
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=18)

fig_b.savefig('figure_4b.svg', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# FIGURE: Per-IGR 100M vs SIM scatter / histogram
# ============================================================

merged_check = results_gdf_100m[['rgi', 'nome_rgi', 'heat_related_attributable_fraction_mean']].copy()
merged_check = merged_check.merge(
    results_gdf_sim[['rgi', 'heat_related_attributable_fraction_mean']].rename(
        columns={'heat_related_attributable_fraction_mean': 'heat_related_af_sim'}
    ),
    on='rgi'
)

# Load population separately (not stored in results_gdf)
population_df = pd.read_parquet('/bp1/geog-tropical/data/BREATHE/igr_population.parquet')
population_df['rgi'] = population_df['rgi'].astype(str)
merged_check = merged_check.merge(population_df[['rgi', 'population_count']], on='rgi')

merged_check['diff'] = (
    merged_check['heat_related_attributable_fraction_mean']
    - merged_check['heat_related_af_sim']
) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(merged_check['diff'], bins=40, edgecolor='k', alpha=0.7)
axes[0].axvline(0, color='red', lw=1.5, ls='--')
axes[0].set_xlabel('100M − SIM attributable fraction')
axes[0].set_ylabel('Number of IGRs')
axes[0].set_title('Distribution of per-IGR differences')
n_lower  = (merged_check['diff'] < 0).sum()
n_higher = (merged_check['diff'] > 0).sum()
axes[0].text(0.02, 0.97, f'Higher in SIM: {n_lower} IGRs\nHigher in 100M: {n_higher} IGRs',
             transform=axes[0].transAxes, va='top', fontsize=11)

sc = axes[1].scatter(
    merged_check['population_count'], merged_check['diff'],
    alpha=0.5, s=20,
    c=merged_check['diff'], cmap='coolwarm', vmin=-20, vmax=20
)
axes[1].axhline(0, color='red', lw=1.5, ls='--')
axes[1].set_xscale('log')
axes[1].set_xlabel('IGR population (log scale)')
axes[1].set_ylabel('100M − SIM attributable fraction')
axes[1].set_title('Difference vs population size')
plt.colorbar(sc, ax=axes[1], label='Difference (pp)')

top_diff = merged_check.loc[merged_check['diff'].idxmax()]
axes[1].annotate(top_diff['nome_rgi'],
                 xy=(top_diff['population_count'], top_diff['diff']),
                 xytext=(15, -10), textcoords='offset points',
                 fontsize=9, color='red',
                 arrowprops=dict(arrowstyle='->', color='red', lw=1))
axes[1].scatter(top_diff['population_count'], top_diff['diff'],
                s=60, color='red', zorder=5, edgecolors='black', linewidths=0.8)

plt.tight_layout()
plt.savefig('igr_fraction_diff.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"IGRs where SIM is higher:  {n_lower} ({n_lower/len(merged_check):.1f}%)")
print(f"IGRs where 100M is higher: {n_higher} ({n_higher/len(merged_check):.1f}%)")
print(f"\nPopulation-weighted mean diff: "
      f"{np.average(merged_check['diff'], weights=merged_check['population_count']):.2f} pp")
print(f"Unweighted mean diff:          {merged_check['diff'].mean():.2f} pp")
print(f"Highest diff IGR: {top_diff['nome_rgi']}, diff: {top_diff['diff']:.1f} pp, "
      f"100M fraction: {top_diff['heat_related_attributable_fraction_mean']:.1f}%")

In [ ]:
# find the highestdiff IGR and print its details
max_diff_igr = merged_check.loc[merged_check['diff'].idxmax()]
print("\nIGR with highest positive difference (100M > SIM):")
print(f"Name: {max_diff_igr['nome_rgi']}")
print(f"100M attributable fraction: {max_diff_igr['heat_related_attributable_fraction_mean']:.2f}%")
print(f"SIM attributable fraction: {max_diff_igr['heat_related_af_sim']:.2f}%")
print(f"Difference (100M - SIM): {max_diff_igr['diff']:.2f} percentage points")
print(f"Population: {max_diff_igr['population_count']}")

In [ ]:
results_gdf_100m[['nome_rgi','rgi','sim_100m_diff']].sort_values('sim_100m_diff', ascending=False).head(10)

In [ ]:
uf_code_to_region = {
    11: 'North', 12: 'North', 13: 'North', 14: 'North', 15: 'North', 16: 'North', 17: 'North',
    21: 'Northeast', 22: 'Northeast', 23: 'Northeast', 24: 'Northeast', 25: 'Northeast',
    26: 'Northeast', 27: 'Northeast', 28: 'Northeast', 29: 'Northeast',
    31: 'Southeast', 32: 'Southeast', 33: 'Southeast', 35: 'Southeast',
    41: 'South', 42: 'South', 43: 'South',
    50: 'Central-West', 51: 'Central-West', 52: 'Central-West', 53: 'Central-West',
}

In [ ]:
# Regional summary: IGR counts where SIM vs 100M is higher
uf_to_region = {
    '11': 'North', '12': 'North', '13': 'North', '14': 'North',
    '15': 'North', '16': 'North', '17': 'North',
    '21': 'Northeast', '22': 'Northeast', '23': 'Northeast', '24': 'Northeast',
    '25': 'Northeast', '26': 'Northeast', '27': 'Northeast', '28': 'Northeast', '29': 'Northeast',
    '31': 'Southeast', '32': 'Southeast', '33': 'Southeast', '35': 'Southeast',
    '41': 'South', '42': 'South', '43': 'South',
    '50': 'Central-West', '51': 'Central-West', '52': 'Central-West', '53': 'Central-West',
}

merged_check['region'] = merged_check['rgi'].astype(str).str[:2].map(uf_to_region)

region_summary = (
    merged_check.dropna(subset=['diff'])
    .groupby('region')
    .apply(lambda g: pd.Series({
        'n_100m_higher':    (g['diff'] > 0).sum(),
        'n_sim_higher':     (g['diff'] < 0).sum(),
        'n_total':          len(g),
        'pct_100m_higher':  (g['diff'] > 0).mean() * 100,
        'pct_sim_higher':   (g['diff'] < 0).mean() * 100,
    }), include_groups=False)
    .reset_index()
    .astype({'n_100m_higher': int, 'n_sim_higher': int, 'n_total': int})
)

print(region_summary.to_string(index=False, float_format=lambda x: f'{x:.1f}'))

In [ ]:
# count population in IGRs where SIM is higher vs 100M is higher
pop_sim_higher = merged_check.loc[merged_check['diff'] < 0, 'population_count'].sum()
pop_100m_higher = merged_check.loc[merged_check['diff'] > 0, 'population_count'].sum()
total_pop = merged_check['population_count'].sum()
print(f"\nPopulation in IGRs where SIM is higher:  {pop_sim_higher} ({pop_sim_higher/total_pop:.1%})")
print(f"Population in IGRs where 100M is higher: {pop_100m_higher} ({pop_100m_higher/total_pop:.1%})")
pop_equal = merged_check.loc[merged_check['diff'] == 0, 'population_count'].sum()
print(f"Population in IGRs where diff == 0: {pop_equal} ({pop_equal/total_pop:.1%})")

n_nan = merged_check['diff'].isna().sum()
pop_nan = merged_check.loc[merged_check['diff'].isna(), 'population_count'].sum()
print(f"IGRs with NaN diff: {n_nan}, population: {pop_nan} ({pop_nan/total_pop:.1%})")

In [ ]:
# def plot_brazil(ax,fig,anom_pq,var=False,change_levels=False):
#     # Define discrete bins
#     cmap = plt.get_cmap("coolwarm")
#     cropped_cmap = cmap(np.linspace(0.1, 1.0, 256))  # skip 0–5% range
#     new_cmap = mcolors.ListedColormap(cropped_cmap)
#     if var == False:
#         if change_levels:
#             bounds = change_levels
#         else:
#             bounds = [0,20,40,60,80,100,120,140,160,180,200]
#         norm = mcolors.BoundaryNorm(bounds, ncolors=256, extend='max')
#     else:
#         # bounds = [0.5,1,1.5,2,2.5,3,3.5,4,4.5,5,5.5,6,6.5,7,7.5,8]
#         if change_levels:
#             bounds = change_levels
#         else:
#             bounds = [-20,-16,-12,-8,-4,0,4,8,12,16,20]
#         norm = mcolors.BoundaryNorm(bounds, ncolors=256, extend='both')

#     # Update the plot with the discrete norm
#     if var == False:
#         anom_pq.plot(column='mortality_anom_per_100000',ax=ax,cmap=new_cmap,norm=norm,vmin=0,vmax=200,legend=False,edgecolor="#f5cca4",linewidth=0.2,alpha=0.9)
#     else:
#         anom_pq.plot(column=var,ax=ax,cmap=new_cmap,norm=norm,vmin=-20,vmax=20,legend=False,edgecolor="#f5cca4",linewidth=0.2,alpha=0.9)
#     # Create horizontal colorbar below the plot
#     sm = plt.cm.ScalarMappable(cmap=new_cmap, norm=norm)
#     sm._A = []  # dummy array for the ScalarMappable
#     cbar_ax = fig.add_axes([0.15, 0.05, 0.35, 0.015])  # [left, bottom, width, height]
    
#     cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal',pad=0, ticks=bounds,extend='both')
#     if var == False:
#         cbar.set_label("Attributable mortality (per 100,000)", fontsize=20,labelpad=10)
#     else:
#         cbar.set_label("Fraction of heat-related deaths attributable to human-induced climate change (percentage of factual deaths)", fontsize=20,labelpad=10)
#     cbar.outline.set_visible(False)
#     cbar.ax.tick_params(labelsize=18)

#     if var == False:
#         ax.set_title("Heat-related mortality attributable to human-induced climate change (per 100,000)",fontsize=22,pad=16)
#     else:
#         ax.set_title("Difference between 100M - SIM \n Fraction of heat-related deaths attributable to human-induced climate change \n (percentage of factual deaths)",fontsize=22,pad=16)
#     ax.spines["left"].set_visible(False)
#     ax.spines["right"].set_visible(False)
#     ax.spines["top"].set_visible(False)
#     ax.spines["bottom"].set_visible(False)
#     ax.set_yticks([])
#     ax.set_xticks([])
#     ax.set_xlabel("",fontsize=14)
#     ax.set_ylabel("",fontsize=14)
    
#     return ax,fig

In [ ]:
# def annotate_capitals(ax, cities, fontsize=18):
#     """Scatter and label Brazilian state capitals on a map axis."""
#     for city in cities:
#         marker = city.get('marker', 'o')
#         s = city.get('s', 14)
#         ax.scatter(city['x'], city['y'], color='black', s=s, marker=marker)
#         ax.annotate(city['name'], xy=(city['ax'], city['ay']), xycoords='data', fontsize=fontsize)

In [ ]:
# def load_simulation(path):
#     """Load a factual or counterfactual simulation Excel file."""
#     df = pd.read_excel(path)
#     # df[df < 0] = 0.1  # Set negative values to zero
#     return df

In [ ]:
# def weighted_percentile(data, weights, percentiles):
#     i = np.argsort(data)
#     d_sorted = data[i]
#     w_sorted = weights[i]
#     # cum_w = np.cumsum(w_sorted)
#     # pct_positions = np.array(percentiles) / 100
#     lower = np.percentile(d_sorted,percentiles[0],method='inverted_cdf',weights=w_sorted)
#     upper = np.percentile(d_sorted,percentiles[1],method='inverted_cdf',weights=w_sorted)
#     return lower,upper

In [ ]:
# # ---------------
# # 1. load in data
# # ---------------
# descriptives = pd.read_excel('/user/home/al18709/breathe/data_analysis/Total_deaths_mean_time_series.xlsx')
# polys = '/bp1/geog-tropical/data/BREATHE/RG2017_rgi_20180911/RG2017_rgi.shp'
# gdf = gpd.read_file(polys)
# population_df = pd.read_parquet(f"/bp1/geog-tropical/data/BREATHE/igr_population.parquet")
# merged = gdf.merge(population_df, on="rgi")

In [ ]:
# # ------------------------------------
# # load in modelled mortality data 100M
# # ------------------------------------

# dir = '/bp1/geog-tropical/data/BREATHE/Simulation_v3/100M/'

# # =============
# # load datasets
# # =============

# # ACCESS-CM2
# for run in range(1,2):
#     # factual
#     globals()[f'accesscm2{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_ACCESS-CM2_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'accesscm2{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_ACCESS-CM2_r{run}i1p1f1.xlsx')

# # ACCESS-ESM1-5
# for run in range(1,4):
#     # factual
#     globals()[f'access{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_ACCESS-ESM1-5_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'access{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_ACCESS-ESM1-5_r{run}i1p1f1.xlsx')

# # BCC-CSM2-MR
# for run in range(1,2):
#     # factual
#     globals()[f'bcc{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_BCC-CSM2-MR_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'bcc{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_BCC-CSM2-MR_r{run}i1p1f1.xlsx')
    
# # CNRM-CM6-1
# for run in range(1,4):
#     # factual
#     globals()[f'cnrm{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_CNRM-CM6-1_r{run}i1p1f2.xlsx')
#     # counterfactual
#     globals()[f'cnrm{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_CNRM-CM6-1_r{run}i1p1f2.xlsx')

# # CanESM5
# for run in range(1,26):
#     if run in [12,16,17,18,19,20,21,24]:
#         continue
#     # factual
#     globals()[f'canesm{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_CanESM5_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'canesm{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_CanESM5_r{run}i1p1f1.xlsx')
    
# # CanESM5 p2
# for run in range(1,26):
#     if run in [3,7,12,13,14,15,17,19,23]:
#         continue
#     # factual
#     globals()[f'canesmp2{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_CanESM5_r{run}i1p2f1.xlsx')
#     # counterfactual
#     globals()[f'canesmp2{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_CanESM5_r{run}i1p2f1.xlsx')

# # FGOALS-g3
# for run in range(1,2):
#     # factual
#     globals()[f'fgoals{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_FGOALS-g3_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'fgoals{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_FGOALS-g3_r{run}i1p1f1.xlsx')

# # GFDL-CM4
# for run in range(1,2):
#     # factual
#     globals()[f'gfdl{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_GFDL-CM4_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'gfdl{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_GFDL-CM4_r{run}i1p1f1.xlsx')

# # GFDL-ESM4
# for run in range(1,2):
#     # factual
#     globals()[f'gfdlesm{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_GFDL-ESM4_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'gfdlesm{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_GFDL-ESM4_r{run}i1p1f1.xlsx')

# # HadGEM3-GC31-LL
# for run in range(1,61):
#     if run in [6,7,8,9,10]:
#         continue
#     # factual
#     globals()[f'hadgem{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_HadGEM3-GC31-LL_r{run}i1p1f3.xlsx')
#     # counterfactual
#     globals()[f'hadgem{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_HadGEM3-GC31-LL_r{run}i1p1f3.xlsx')

# # IPSL-CM6A-LR
# for run in range(1,7):
#     if run == 3:
#         continue
#     # factual
#     globals()[f'ipsl{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_IPSL-CM6A-LR_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'ipsl{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_IPSL-CM6A-LR_r{run}i1p1f1.xlsx')

# # MIROC6
# for run in range(1,3):
#     # factual
#     globals()[f'miroc{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_MIROC6_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'miroc{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_MIROC6_r{run}i1p1f1.xlsx')

# # MRI-ESM2-0
# for run in range(1,6):
#     if run == 4:
#         continue
#     # factual
#     globals()[f'mri{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_MRI-ESM2-0_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'mri{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_MRI-ESM2-0_r{run}i1p1f1.xlsx')
    
# # NorESM2-LM
# for run in range(1,4):
#     # factual
#     globals()[f'noresm{run}_factual_100m'] = pd.read_excel(dir + f'FACTUAL_AN_COHORT_noRH_tas_NorESM2-LM_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'noresm{run}_counterfactual_100m'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_COHORT_noRH_tas_NorESM2-LM_r{run}i1p1f1.xlsx')

In [ ]:
# factual_vars = {name: obj for name, obj in globals().items() 
#                 if name.endswith('_factual_100m') and isinstance(obj, pd.DataFrame)}

# negative_count = 0
# total_count = 0
# negative_sims = []

# for name, df in factual_vars.items():
#     total_count += 1
#     if (df < 0).any().any():
#         negative_count += 1
#         negative_sims.append(name)

# print(f"Simulations with any negative AN: {negative_count} / {total_count}")
# print(f"\nNegative simulations:")
# for sim in negative_sims:
#     print(f"  {sim}")

In [ ]:
# # -----------------------------------
# # load in modelled mortality data SIM
# # -----------------------------------
# # sim
# dir = '/bp1/geog-tropical/data/BREATHE/Simulation_v3/SIM/'


# # ACCESS-CM2
# for run in range(1,2):
#     # factual
#     globals()[f'accesscm2{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_ACCESS-CM2_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'accesscm2{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_ACCESS-CM2_r{run}i1p1f1.xlsx')

# # ACCESS-ESM1-5
# for run in range(1,4):
#     # factual
#     globals()[f'access{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_ACCESS-ESM1-5_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'access{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_ACCESS-ESM1-5_r{run}i1p1f1.xlsx')

# # BCC-CSM2-MR
# for run in range(1,2):
#     # factual
#     globals()[f'bcc{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_BCC-CSM2-MR_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'bcc{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_BCC-CSM2-MR_r{run}i1p1f1.xlsx')
    
# # CNRM-CM6-1
# for run in range(1,4):
#     # factual
#     globals()[f'cnrm{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_CNRM-CM6-1_r{run}i1p1f2.xlsx')
#     # counterfactual
#     globals()[f'cnrm{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_CNRM-CM6-1_r{run}i1p1f2.xlsx')

# # CanESM5
# for run in range(1,26):
#     if run in [12,16,17,18,19,20,21,24]:
#         continue
#     # factual
#     globals()[f'canesm{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_CanESM5_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'canesm{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_CanESM5_r{run}i1p1f1.xlsx')
    
# # CanESM5 p2
# for run in range(1,26):
#     if run in [3,7,12,13,14,15,17,19,23]:
#         continue
#     # factual
#     globals()[f'canesmp2{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_CanESM5_r{run}i1p2f1.xlsx')
#     # counterfactual
#     globals()[f'canesmp2{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_CanESM5_r{run}i1p2f1.xlsx')

# # FGOALS-g3
# for run in range(1,2):
#     # factual
#     globals()[f'fgoals{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_FGOALS-g3_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'fgoals{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_FGOALS-g3_r{run}i1p1f1.xlsx')

# # GFDL-CM4
# for run in range(1,2):
#     # factual
#     globals()[f'gfdl{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_GFDL-CM4_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'gfdl{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_GFDL-CM4_r{run}i1p1f1.xlsx')

# # GFDL-ESM4
# for run in range(1,2):
#     # factual
#     globals()[f'gfdlesm{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_GFDL-ESM4_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'gfdlesm{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_GFDL-ESM4_r{run}i1p1f1.xlsx')

# # HadGEM3-GC31-LL
# for run in range(1,61):
#     if run in [6,7,8,9,10]:
#         continue
#     # factual
#     globals()[f'hadgem{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_HadGEM3-GC31-LL_r{run}i1p1f3.xlsx')
#     # counterfactual
#     globals()[f'hadgem{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_HadGEM3-GC31-LL_r{run}i1p1f3.xlsx')

# # IPSL-CM6A-LR
# for run in range(1,7):
#     if run == 3:
#         continue
#     # factual
#     globals()[f'ipsl{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_IPSL-CM6A-LR_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'ipsl{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_IPSL-CM6A-LR_r{run}i1p1f1.xlsx')

# # MIROC6
# for run in range(1,3):
#     # factual
#     globals()[f'miroc{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_MIROC6_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'miroc{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_MIROC6_r{run}i1p1f1.xlsx')

# # MRI-ESM2-0
# for run in range(1,6):
#     if run == 4:
#         continue
#     # factual
#     globals()[f'mri{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_MRI-ESM2-0_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'mri{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_MRI-ESM2-0_r{run}i1p1f1.xlsx')
    
# # NorESM2-LM
# for run in range(1,4):
#     # factual
#     globals()[f'noresm{run}_factual_sim'] = pd.read_excel(dir + f'FACTUAL_AN_SIM_noRH_tas_NorESM2-LM_r{run}i1p1f1.xlsx')
#     # counterfactual
#     globals()[f'noresm{run}_counterfactual_sim'] = pd.read_excel(dir + f'COUNTERFACTUAL_AN_SIM_noRH_tas_NorESM2-LM_r{run}i1p1f1.xlsx')





In [ ]:
# # define model families and their available runs
# models = {
#     'accesscm2': [1],
#     'access':    list(range(1, 4)),
#     'bcc':       [1],
#     'cnrm':      list(range(1, 4)),
#     'canesm':    [r for r in range(1, 26) if r not in [12, 16, 17, 18, 19, 20, 21, 24]],
#     'canesmp2':  [r for r in range(1, 26) if r not in [3, 7, 12, 13, 14, 15, 17, 19,23]],
#     'fgoals':    [1],
#     'gfdl':      [1],
#     'gfdlesm':   [1],
#     'hadgem':    [r for r in range(1, 61) if r not in [6, 7, 8, 9, 10]],
#     'ipsl':      [r for r in range(1, 7) if r != 3],
#     'miroc':     list(range(1, 3)),
#     'mri':       [r for r in range(1, 6) if r != 4],
#     'noresm':    list(range(1, 4)),
# }



In [ ]:
# # sort everything to keep order
# for model, runs in models.items():
#     for run in runs:
#         globals()[f'{model}{run}_factual_100m'] = globals()[f'{model}{run}_factual_100m'].sort_index(axis=1)
#         globals()[f'{model}{run}_counterfactual_100m'] = globals()[f'{model}{run}_counterfactual_100m'].sort_index(axis=1)
#         globals()[f'{model}{run}_factual_sim'] = globals()[f'{model}{run}_factual_sim'].sort_index(axis=1)
#         globals()[f'{model}{run}_counterfactual_sim'] = globals()[f'{model}{run}_counterfactual_sim'].sort_index(axis=1)

In [ ]:
# for model, runs in models.items():
     
#     print(np.sum(np.isnan([globals()[f'{model}{run}_factual_100m'] for run in runs])))

    

In [ ]:
# # -----------------------------------------
# # 2. for each model calculate anomaly pairs
# # -----------------------------------------

# # ====
# # 100m
# # ====

# # for each model family, compute anomaly pairs per IGR
# for model, runs in models.items():
    

    
#     globals()[f'{model}_factual_list'] = [globals()[f'{model}{run}_factual_100m'] for run in runs]
#     globals()[f'{model}_cfactual_list'] = [globals()[f'{model}{run}_counterfactual_100m'] for run in runs]
    
#     # mask negative values
#     for df in globals()[f'{model}_factual_list']:
#         # df[df < 0] = 0.1
#         df.mask(df <= 0, inplace=True)
#     for df in globals()[f'{model}_cfactual_list']:
#         # df[df < 0] = 0.1
#         df.mask(df <= 0, inplace=True)
    
#     for igr in globals()[f'{model}{runs[0]}_factual_100m'].columns:
#         scaling_factor = 1000 * len(runs)
#         # igr_factuals = np.zeros((scaling_factor,))
#         # igr_cfactuals = np.zeros((scaling_factor,))
#         # igr_anomalies = np.zeros((scaling_factor,))
#         # igr_hr_fraction = np.zeros((scaling_factor,))
#         igr_factuals = np.full((scaling_factor,), np.nan)
#         igr_cfactuals = np.full((scaling_factor,), np.nan)
#         igr_anomalies = np.full((scaling_factor,), np.nan)
#         igr_hr_fraction = np.full((scaling_factor,), np.nan)
#         for i in range(len(runs)):
#             factual = globals()[f'{model}_factual_list'][i][igr]
#             cfactual = globals()[f'{model}_cfactual_list'][i][igr]
#             anomaly = factual - cfactual
#             igr_anomalies[i*1000:(i+1)*1000] = anomaly
#             igr_factuals[i*1000:(i+1)*1000] = factual
#             igr_cfactuals[i*1000:(i+1)*1000] = cfactual
#             igr_hr_fraction[i*1000:(i+1)*1000] = anomaly / factual
#         globals()[f'{model}_anomaly_pairs_{igr}'] = igr_anomalies
#         globals()[f'{model}_factuals_{igr}'] = igr_factuals
#         globals()[f'{model}_cfactuals_{igr}'] = igr_cfactuals
#         globals()[f'{model}_hr_fraction_{igr}'] = igr_hr_fraction

# # ===
# # SIM
# # ===

# for model, runs in models.items():
    
#     globals()[f'{model}_factual_list_sim'] = [globals()[f'{model}{run}_factual_sim'] for run in runs]
#     globals()[f'{model}_cfactual_list_sim'] = [globals()[f'{model}{run}_counterfactual_sim'] for run in runs]
    
#     # mask negative values as mask not 0.1
#     for df in globals()[f'{model}_factual_list_sim']:
#         df.mask(df <= 0, inplace=True)
#         # df[df < 0] = 0.1
#     for df in globals()[f'{model}_cfactual_list_sim']:
#         df.mask(df <= 0, inplace=True)
#         # df[df < 0] = 0.1

#     for igr in globals()[f'{model}{runs[0]}_factual_sim'].columns:
#         scaling_factor = 1000 * len(runs)
#         # igr_factuals = np.zeros((scaling_factor,))
#         # igr_cfactuals = np.zeros((scaling_factor,))
#         # igr_anomalies = np.zeros((scaling_factor,))
#         # igr_hr_fraction = np.zeros((scaling_factor,))
#         igr_factuals = np.full((scaling_factor,), np.nan)
#         igr_cfactuals = np.full((scaling_factor,), np.nan)
#         igr_anomalies = np.full((scaling_factor,), np.nan)
#         igr_hr_fraction = np.full((scaling_factor,), np.nan)
#         for i in range(len(runs)):
#             factual = globals()[f'{model}_factual_list_sim'][i][igr]
#             cfactual = globals()[f'{model}_cfactual_list_sim'][i][igr]
#             anomaly = factual - cfactual
#             igr_anomalies[i*1000:(i+1)*1000] = anomaly
#             igr_factuals[i*1000:(i+1)*1000] = factual
#             igr_cfactuals[i*1000:(i+1)*1000] = cfactual
#             igr_hr_fraction[i*1000:(i+1)*1000] = anomaly / factual
#         globals()[f'{model}_anomaly_pairs_sim_{igr}'] = igr_anomalies
#         globals()[f'{model}_factuals_sim_{igr}'] = igr_factuals
#         globals()[f'{model}_cfactuals_sim_{igr}'] = igr_cfactuals
#         globals()[f'{model}_hr_fraction_sim_{igr}'] = igr_hr_fraction




In [ ]:
# df

In [ ]:
# # # -----------------------------
# # # 3. calculate aggregate totals
# # # -----------------------------

# uf_code_to_region = {
#     11: 'North', 12: 'North', 13: 'North', 14: 'North', 15: 'North', 16: 'North', 17: 'North',
#     21: 'Northeast', 22: 'Northeast', 23: 'Northeast', 24: 'Northeast', 25: 'Northeast',
#     26: 'Northeast', 27: 'Northeast', 28: 'Northeast', 29: 'Northeast',
#     31: 'Southeast', 32: 'Southeast', 33: 'Southeast', 35: 'Southeast',
#     41: 'South', 42: 'South', 43: 'South',
#     50: 'Central-West', 51: 'Central-West', 52: 'Central-West', 53: 'Central-West',
# }

# regions = ['North', 'Northeast', 'Southeast', 'South', 'Central-West']

# def get_region_cols(df, region):
#     return [c for c in df.columns if uf_code_to_region.get(int(str(c)[:2])) == region]

# def calc_aggregates(factual_list, cfactual_list):
#     results = {r: {'factuals': [], 'cfactuals': [], 'anomalies': [], 'hr_fraction': []} for r in regions + ['Brazil']}
#     for factual_df, cfactual_df in zip(factual_list, cfactual_list):
#         for region in regions:
#             cols = get_region_cols(factual_df, region)
#             f = factual_df[cols].sum(axis=1, skipna=True)
#             c = cfactual_df[cols].sum(axis=1, skipna=True)
#             results[region]['factuals'].append(f)
#             results[region]['cfactuals'].append(c)
#             results[region]['anomalies'].append(f - c)
#             results[region]['hr_fraction'].append((f - c) / f)
#         # Brazil total
#         f = factual_df.sum(axis=1, skipna=True)
#         c = cfactual_df.sum(axis=1, skipna=True)
#         results['Brazil']['factuals'].append(f)
#         results['Brazil']['cfactuals'].append(c)
#         results['Brazil']['anomalies'].append(f - c)
#         results['Brazil']['hr_fraction'].append((f - c) / f)
#     return results


# for model in models.keys():
#     globals()[f'{model}_results'] = calc_aggregates(globals()[f'{model}_factual_list'], globals()[f'{model}_cfactual_list'])
#     globals()[f'{model}_results_sim'] = calc_aggregates(globals()[f'{model}_factual_list_sim'], globals()[f'{model}_cfactual_list_sim'])



In [ ]:
# np.sum(np.isnan(access_factual_list))

In [ ]:
# np.sum(np.isinf(access_results['North']['factuals']))

In [ ]:
# # Brazil

# from functools import reduce

# all_brazil_factuals  = np.concatenate(reduce(lambda a, b: a + b, [globals()[f'{model}_results']['Brazil']['factuals']  for model in models.keys()]))
# all_brazil_cfactuals = np.concatenate(reduce(lambda a, b: a + b, [globals()[f'{model}_results']['Brazil']['cfactuals'] for model in models.keys()]))
# all_brazil_anomalies = np.concatenate(reduce(lambda a, b: a + b, [globals()[f'{model}_results']['Brazil']['anomalies'] for model in models.keys()]))
# all_brazil_hr_fraction = np.concatenate(reduce(lambda a, b: a + b, [globals()[f'{model}_results']['Brazil']['hr_fraction'] for model in models.keys()]))

# all_brazil_factuals_sim  = np.concatenate(reduce(lambda a, b: a + b, [globals()[f'{model}_results_sim']['Brazil']['factuals']  for model in models.keys()]))
# all_brazil_cfactuals_sim = np.concatenate(reduce(lambda a, b: a + b, [globals()[f'{model}_results_sim']['Brazil']['cfactuals'] for model in models.keys()]))
# all_brazil_anomalies_sim = np.concatenate(reduce(lambda a, b: a + b, [globals()[f'{model}_results_sim']['Brazil']['anomalies'] for model in models.keys()]))
# all_brazil_hr_fraction_sim = np.concatenate(reduce(lambda a, b: a + b, [globals()[f'{model}_results_sim']['Brazil']['hr_fraction'] for model in models.keys()]))


# from functools import reduce

# def concat_region(region, key,sim=False):
#     if sim == False:
#         return np.concatenate(reduce(lambda a, b: a + b, [globals()[f'{model}_results'][region][key] for model in models.keys()]))
#     else:
#         return np.concatenate(reduce(lambda a, b: a + b, [globals()[f'{model}_results_sim'][region][key] for model in models.keys()]))

# # north
# north_factuals  = concat_region('North', 'factuals')
# north_cfactuals = concat_region('North', 'cfactuals')
# north_anomalies = concat_region('North', 'anomalies')
# north_hr_fraction = concat_region('North', 'hr_fraction')
# # northeast
# northeast_factuals  = concat_region('Northeast', 'factuals')
# northeast_cfactuals = concat_region('Northeast', 'cfactuals')
# northeast_anomalies = concat_region('Northeast', 'anomalies')
# northeast_hr_fraction = concat_region('Northeast', 'hr_fraction')

# # southeast
# southeast_factuals  = concat_region('Southeast', 'factuals')
# southeast_cfactuals = concat_region('Southeast', 'cfactuals')
# southeast_anomalies = concat_region('Southeast', 'anomalies')
# southeast_hr_fraction = concat_region('Southeast', 'hr_fraction')

# # south
# south_factuals  = concat_region('South', 'factuals')
# south_cfactuals = concat_region('South', 'cfactuals')
# south_anomalies = concat_region('South', 'anomalies')
# south_hr_fraction = concat_region('South', 'hr_fraction')

# # central-west
# centralwest_factuals  = concat_region('Central-West', 'factuals')
# centralwest_cfactuals = concat_region('Central-West', 'cfactuals')
# centralwest_anomalies = concat_region('Central-West', 'anomalies')
# centralwest_hr_fraction = concat_region('Central-West', 'hr_fraction')


# # sim

# # north
# north_factuals_sim  = concat_region('North', 'factuals',sim=True)
# north_cfactuals_sim = concat_region('North', 'cfactuals',sim=True)
# north_anomalies_sim = concat_region('North', 'anomalies',sim=True)
# north_hr_fraction_sim = concat_region('North', 'hr_fraction',sim=True)

# # northeast
# northeast_factuals_sim  = concat_region('Northeast', 'factuals',sim=True)
# northeast_cfactuals_sim = concat_region('Northeast', 'cfactuals',sim=True)
# northeast_anomalies_sim = concat_region('Northeast', 'anomalies',sim=True)
# northeast_hr_fraction_sim = concat_region('Northeast', 'hr_fraction',sim=True)

# # southeast
# southeast_factuals_sim  = concat_region('Southeast', 'factuals',sim=True)
# southeast_cfactuals_sim = concat_region('Southeast', 'cfactuals',sim=True)
# southeast_anomalies_sim = concat_region('Southeast', 'anomalies',sim=True)
# southeast_hr_fraction_sim = concat_region('Southeast', 'hr_fraction',sim=True)

# # south
# south_factuals_sim  = concat_region('South', 'factuals',sim=True)
# south_cfactuals_sim = concat_region('South', 'cfactuals',sim=True)
# south_anomalies_sim = concat_region('South', 'anomalies',sim=True)
# south_hr_fraction_sim = concat_region('South', 'hr_fraction',sim=True)

# # central-west
# centralwest_factuals_sim  = concat_region('Central-West', 'factuals',sim=True)
# centralwest_cfactuals_sim = concat_region('Central-West', 'cfactuals',sim=True)
# centralwest_anomalies_sim = concat_region('Central-West', 'anomalies',sim=True)
# centralwest_hr_fraction_sim = concat_region('Central-West', 'hr_fraction',sim=True)



In [ ]:
# # combine datasets and assign weighting
# for model in models.keys():
#     globals()[f'{model}'] = globals()[f'{model}_anomaly_pairs_{igr}']
#     globals()[f'{model}_sim'] = globals()[f'{model}_anomaly_pairs_sim_{igr}']
#     globals()[f'{model}_hr_fraction'] = globals()[f'{model}_hr_fraction_{igr}']
#     globals()[f'{model}_hr_fraction_sim'] = globals()[f'{model}_hr_fraction_sim_{igr}']
    

# weights_new = np.concatenate([np.full_like(globals()[f'{model}_anomaly_pairs_{igr}'], len(models) / len(globals()[f'{model}_anomaly_pairs_{igr}']), dtype=float) for model in models.keys()])
# # apply a mask for simulations where deaths are 0 or negative and masked


# # weights_new = np.concatenate([
# #     np.where(
# #         np.isnan(globals()[f'{model}_hr_fraction_{igr}']),
# #         0.0,
# #         np.full_like(globals()[f'{model}_hr_fraction_{igr}'], len(models) / np.sum(~np.isnan(globals()[f'{model}_hr_fraction_{igr}'])), dtype=float)
# #     )
# #     for model in models.keys()
# # ])



In [ ]:


# # combine anomaly pairs and calculate mean and perentiles
# mortality_estimates = pd.DataFrame({'rgi':[],'mean_attributable_mortality':[],'lower_ci':[],'upper_ci':[]})
# for igr in globals()[f'{model}1_factual_100m'].columns:
#     combined = np.concat([globals()[f'{model}_anomaly_pairs_{igr}'] for model in models.keys()])
#     combined_factual = np.concat([globals()[f'{model}_factuals_{igr}'] for model in models.keys()])
#     combined_hr_fraction = np.concat([globals()[f'{model}_hr_fraction_{igr}'] for model in models.keys()])
#     # igr_mean = np.average(combined,weights=weights_new)
#     # igr_factual_mean = np.average(combined_factual,weights=weights_new)
#     # igr_hr_fraction_mean = np.average(combined_hr_fraction,weights=weights_new)
#     # percentile25,percentile975 = weighted_percentile(combined,weights_new,[2.5,97.5])
#     # percentile25_factual,percentile975_factual = weighted_percentile(combined_factual,weights_new,[2.5,97.5])
#     # percentile25_hr_fraction,percentile975_hr_fraction = weighted_percentile(combined_hr_fraction,weights_new,[2.5,97.5])
    
#     igr_mean = np.average(np.nan_to_num(combined),weights=weights_new)
#     igr_factual_mean = np.average(np.nan_to_num(combined_factual),weights=weights_new)
#     igr_hr_fraction_mean = np.average(np.nan_to_num(combined_hr_fraction),weights=weights_new)
#     percentile25,percentile975 = weighted_percentile(np.nan_to_num(combined),weights_new,[2.5,97.5])
#     percentile25_factual,percentile975_factual = weighted_percentile(np.nan_to_num(combined_factual),weights_new,[2.5,97.5])
#     percentile25_hr_fraction,percentile975_hr_fraction = weighted_percentile(np.nan_to_num(combined_hr_fraction),weights_new,[2.5,97.5])


    
    
#     new_row = pd.DataFrame([{
#     'rgi': igr,
#     'mean_attributable_mortality': igr_mean,
#     'mean_factual_mortality': igr_factual_mean,
#     'mean_hr_fraction': igr_hr_fraction_mean,
#     'lower_ci': percentile25,
#     'upper_ci': percentile975,
#     'lower_ci_factual': percentile25_factual,
#     'upper_ci_factual': percentile975_factual,
#     'lower_ci_hr_fraction': percentile25_hr_fraction,
#     'upper_ci_hr_fraction': percentile975_hr_fraction
#     }])
#     mortality_estimates = pd.concat([mortality_estimates, new_row], ignore_index=True)
    

# mortality_estimates_sim = pd.DataFrame({'rgi':[],'mean_attributable_mortality':[],'lower_ci':[],'upper_ci':[]})
# # for igr in gdfl_factual_sim.columns:
# for igr in globals()[f'{model}1_factual_sim'].columns:
#     combined = np.concat([globals()[f'{model}_anomaly_pairs_sim_{igr}'] for model in models.keys()])
#     combined_factual = np.concat([globals()[f'{model}_factuals_sim_{igr}'] for model in models.keys()])
#     combined_hr_fraction = np.concat([globals()[f'{model}_hr_fraction_sim_{igr}'] for model in models.keys()])
    
#     # combined = np.concat([globals()[f'canesm_anomaly_pairs_sim_{igr}'],globals()[f'mri_anomaly_pairs_sim_{igr}'],globals()[f'gdfl_anomaly_pairs_sim_{igr}']])
#     # combined_factual = np.concat([globals()[f'canesm_factuals_sim_{igr}'],globals()[f'mri_factuals_sim_{igr}'],globals()[f'gdfl_factuals_sim_{igr}']])
#     # igr_mean = np.average(combined,weights=weights_new)
#     # igr_factual_mean = np.average(combined_factual,weights=weights_new)
#     # igr_hr_fraction_mean = np.average(combined_hr_fraction,weights=weights_new)
#     # percentile25,percentile975 = weighted_percentile(combined,weights_new,[2.5,97.5])
#     # percentile25_factual,percentile975_factual = weighted_percentile(combined_factual,weights_new,[2.5,97.5])
#     # percentile25_hr_fraction,percentile975_hr_fraction = weighted_percentile(combined_hr_fraction,weights_new,[2.5,97.5])
    
#     igr_mean = np.average(np.nan_to_num(combined),weights=weights_new)
#     igr_factual_mean = np.average(np.nan_to_num(combined_factual),weights=weights_new)
#     igr_hr_fraction_mean = np.average(np.nan_to_num(combined_hr_fraction),weights=weights_new)
#     percentile25,percentile975 = weighted_percentile(np.nan_to_num(combined),weights_new,[2.5,97.5])
#     percentile25_factual,percentile975_factual = weighted_percentile(np.nan_to_num(combined_factual),weights_new,[2.5,97.5])
#     percentile25_hr_fraction,percentile975_hr_fraction = weighted_percentile(np.nan_to_num(combined_hr_fraction),weights_new,[2.5,97.5])
    
    
#     new_row = pd.DataFrame([{
#     'rgi': igr,
#     'mean_attributable_mortality': igr_mean,
#     'mean_factual_mortality': igr_factual_mean,
#     'mean_hr_fraction': igr_hr_fraction_mean,
#     'lower_ci': percentile25,
#     'upper_ci': percentile975,
#     'lower_ci_factual': percentile25_factual,
#     'upper_ci_factual': percentile975_factual,
#     'lower_ci_hr_fraction': percentile25_hr_fraction,
#     'upper_ci_hr_fraction': percentile975_hr_fraction
#     }])
#     mortality_estimates_sim = pd.concat([mortality_estimates_sim, new_row], ignore_index=True)


In [ ]:


# # claculate brazil wide statistics as per reviewer comment 100M
# all_brazil_attributable_mean = np.average(all_brazil_anomalies,weights=weights_new)
# all_brazil_25,all_brazil_975 = weighted_percentile(all_brazil_anomalies,weights_new,[2.5,97.5])
# # all_brazil_af_climate_change = np.average(all_brazil_hr_fraction,weights=weights_new)
# # all_brazil_af_climate_change_25,all_brazil_af_climate_change_975 = weighted_percentile(all_brazil_hr_fraction,weights_new,[2.5,97.5])
# all_brazil_factual_mean = np.average(all_brazil_factuals,weights=weights_new)
# all_brazil_25_factual,all_brazil_975_factual = weighted_percentile(all_brazil_factuals,weights_new,[2.5,97.5])


# north_attributable_mean = np.average(north_anomalies,weights=weights_new)
# north_25,north_975 = weighted_percentile(north_anomalies,weights_new,[2.5,97.5])
# northeast_attributable_mean = np.average(northeast_anomalies,weights=weights_new)
# northeast_25,northeast_975 = weighted_percentile(northeast_anomalies,weights_new,[2.5,97.5])
# southeast_attributable_mean = np.average(southeast_anomalies,weights=weights_new)
# southeast_25,southeast_975 = weighted_percentile(southeast_anomalies,weights_new,[2.5,97.5])
# south_attributable_mean = np.average(south_anomalies,weights=weights_new)
# south_25,south_975 = weighted_percentile(south_anomalies,weights_new,[2.5,97.5])
# centralwest_attributable_mean = np.average(centralwest_anomalies,weights=weights_new)
# centralwest_25,centralwest_975 = weighted_percentile(centralwest_anomalies,weights_new,[2.5,97.5])

# af_climate_change = all_brazil_anomalies / all_brazil_factuals
# all_brazil_af_climate_change = np.average(af_climate_change,weights=weights_new)
# all_brazil_af_climate_change_25,all_brazil_af_climate_change_975 = weighted_percentile(af_climate_change,weights_new,[2.5,97.5])

# af_cc_north = north_anomalies / north_factuals
# north_af_cc = np.average(af_cc_north,weights=weights_new)
# north_af_cc_25,north_af_cc_975 = weighted_percentile(af_cc_north,weights_new,[2.5,97.5])

# af_cc_northeast = northeast_anomalies / northeast_factuals
# northeast_af_cc = np.average(af_cc_northeast,weights=weights_new)
# northeast_af_cc_25,northeast_af_cc_975 = weighted_percentile(af_cc_northeast,weights_new,[2.5,97.5])

# af_cc_southeast = southeast_anomalies / southeast_factuals
# southeast_af_cc = np.average(af_cc_southeast,weights=weights_new)
# southeast_af_cc_25,southeast_af_cc_975 = weighted_percentile(af_cc_southeast,weights_new,[2.5,97.5])

# af_cc_south = south_anomalies / south_factuals
# south_af_cc = np.average(af_cc_south,weights=weights_new)
# south_af_cc_25,south_af_cc_975 = weighted_percentile(af_cc_south,weights_new,[2.5,97.5])

# af_cc_centralwest = centralwest_anomalies / centralwest_factuals
# centralwest_af_cc = np.average(af_cc_centralwest,weights=weights_new)
# centralwest_af_cc_25,centralwest_af_cc_975 = weighted_percentile(af_cc_centralwest,weights_new,[2.5,97.5])


# # SIM
# all_brazil_attributable_mean_sim = np.average(all_brazil_anomalies_sim,weights=weights_new)
# all_brazil_25_sim,all_brazil_975_sim = weighted_percentile(all_brazil_anomalies_sim,weights_new,[2.5,97.5])

# all_brazil_factual_mean_sim = np.average(all_brazil_factuals_sim,weights=weights_new)
# all_brazil_25_factual_sim,all_brazil_975_factual_sim = weighted_percentile(all_brazil_factuals_sim,weights_new,[2.5,97.5])


# af_climate_change_sim = all_brazil_anomalies_sim / all_brazil_factuals_sim
# all_brazil_af_climate_change_sim = np.average(af_climate_change_sim,weights=weights_new)
# all_brazil_af_climate_change_25_sim,all_brazil_af_climate_change_975_sim = weighted_percentile(af_climate_change_sim,weights_new,[2.5,97.5])

# north_attributable_mean_sim = np.average(north_anomalies_sim,weights=weights_new)
# north_25_sim,north_975_sim = weighted_percentile(north_anomalies_sim,weights_new,[2.5,97.5])
# northeast_attributable_mean_sim = np.average(northeast_anomalies_sim,weights=weights_new)
# northeast_25_sim,northeast_975_sim = weighted_percentile(northeast_anomalies_sim,weights_new,[2.5,97.5])
# southeast_attributable_mean_sim = np.average(southeast_anomalies_sim,weights=weights_new)
# southeast_25_sim,southeast_975_sim = weighted_percentile(southeast_anomalies_sim,weights_new,[2.5,97.5])
# south_attributable_mean_sim = np.average(south_anomalies_sim,weights=weights_new)
# south_25_sim,south_975_sim = weighted_percentile(south_anomalies_sim,weights_new,[2.5,97.5])
# centralwest_attributable_mean_sim = np.average(centralwest_anomalies_sim,weights=weights_new)
# centralwest_25_sim,centralwest_975_sim = weighted_percentile(centralwest_anomalies_sim,weights_new,[2.5,97.5])

# af_cc_north_sim = north_anomalies_sim / north_factuals_sim
# north_af_cc_sim = np.average(af_cc_north_sim,weights=weights_new)
# north_af_cc_25_sim,north_af_cc_975_sim = weighted_percentile(af_cc_north_sim,weights_new,[2.5,97.5])
# af_cc_northeast_sim = northeast_anomalies_sim / northeast_factuals_sim
# northeast_af_cc_sim = np.average(af_cc_northeast_sim,weights=weights_new)
# northeast_af_cc_25_sim,northeast_af_cc_975_sim = weighted_percentile(af_cc_northeast_sim,weights_new,[2.5,97.5])
# af_cc_southeast_sim = southeast_anomalies_sim / southeast_factuals_sim
# southeast_af_cc_sim = np.average(af_cc_southeast_sim,weights=weights_new)
# southeast_af_cc_25_sim,southeast_af_cc_975_sim = weighted_percentile(af_cc_southeast_sim,weights_new,[2.5,97.5])
# af_cc_south_sim = south_anomalies_sim / south_factuals_sim
# south_af_cc_sim = np.average(af_cc_south_sim,weights=weights_new)
# south_af_cc_25_sim,south_af_cc_975_sim = weighted_percentile(af_cc_south_sim,weights_new,[2.5,97.5])
# af_cc_centralwest_sim = centralwest_anomalies_sim / centralwest_factuals_sim
# centralwest_af_cc_sim = np.average(af_cc_centralwest_sim,weights=weights_new)
# centralwest_af_cc_25_sim,centralwest_af_cc_975_sim = weighted_percentile(af_cc_centralwest_sim,weights_new,[2.5,97.5])


In [ ]:
# print('100M')
# print('Number of people who have died from heat-related causes between 2000-2018 in Brazil:')
# print(all_brazil_factual_mean)
# print('per year:',all_brazil_factual_mean/19)

# print('with confidence intervals:')
# print(all_brazil_25)
# print(all_brazil_factual_mean)
# print(all_brazil_975)


# print('\n SIM')
# print('Number of people who have died from heat-related causes between 2000-2018 in Brazil:')
# print(all_brazil_factual_mean_sim)
# print('per year:',all_brazil_factual_mean_sim/19)

# print('with confidence intervals:')
# print(all_brazil_25_sim)
# print(all_brazil_factual_mean_sim)
# print(all_brazil_975_sim)

In [ ]:
# print('100M')
# print('fraction of heat-related deaths in Brazil attributable to climate change:')
# print(all_brazil_af_climate_change*100)
# print('with confidence intervals:')

# print(all_brazil_af_climate_change_25*100)
# print(all_brazil_af_climate_change*100)
# print(all_brazil_af_climate_change_975*100)

# print('\n SIM')
# print('fraction of heat-related deaths in Brazil attributable to climate change:')
# print(all_brazil_af_climate_change_sim*100)
# print('with confidence intervals:')

# print(all_brazil_af_climate_change_25_sim*100)
# print(all_brazil_af_climate_change_sim*100)
# print(all_brazil_af_climate_change_975_sim*100)

In [ ]:
# print('100M')
# print('Number of people who have died from heat-related causes as a result of climate change between 2000-2018 in Brazil:')
# print(all_brazil_attributable_mean)
# print('per year:',all_brazil_attributable_mean/19)

# print('with confidence intervals:')
# print(all_brazil_25)
# print(all_brazil_attributable_mean)
# print(all_brazil_975)


# print('\n SIM')
# print('Number of people who have died from heat-related causes as a result of climate change between 2000-2018 in Brazil:')
# print(all_brazil_attributable_mean_sim)
# print('per year:',all_brazil_attributable_mean_sim/19)

# print('with confidence intervals:')
# print(all_brazil_25_sim)
# print(all_brazil_attributable_mean_sim)
# print(all_brazil_975_sim)



In [ ]:
# import numpy as np

# print("=" * 60)
# print("CONSISTENCY CHECK: pre-computed vs re-derived hr_fraction")
# print("=" * 60)

# regions_to_check = {
#     'Brazil':       (all_brazil_hr_fraction,      af_climate_change),
#     'North':        (north_hr_fraction,            af_cc_north),
#     'Northeast':    (northeast_hr_fraction,        af_cc_northeast),
#     'Southeast':    (southeast_hr_fraction,        af_cc_southeast),
#     'South':        (south_hr_fraction,            af_cc_south),
#     'Central-West': (centralwest_hr_fraction,      af_cc_centralwest),
# }

# regions_to_check_sim = {
#     'Brazil':       (all_brazil_hr_fraction_sim,   af_climate_change_sim),
#     'North':        (north_hr_fraction_sim,        af_cc_north_sim),
#     'Northeast':    (northeast_hr_fraction_sim,    af_cc_northeast_sim),
#     'Southeast':    (southeast_hr_fraction_sim,    af_cc_southeast_sim),
#     'South':        (south_hr_fraction_sim,        af_cc_south_sim),
#     'Central-West': (centralwest_hr_fraction_sim,  af_cc_centralwest_sim),
# }

# def check_region(name, precomputed, rederived, label="100M"):
#     same_length = len(precomputed) == len(rederived)
#     if not same_length:
#         print(f"  [{label}] {name}: LENGTH MISMATCH — {len(precomputed)} vs {len(rederived)}")
#         return

#     close = np.allclose(precomputed, rederived, equal_nan=True)
#     max_diff = np.nanmax(np.abs(precomputed - rederived))
#     mean_precomp  = np.average(precomputed, weights=weights_new)
#     mean_rederived = np.average(rederived,  weights=weights_new)

#     status = "OK" if close else "MISMATCH"
#     print(f"\n  [{label}] {name}: {status}")
#     print(f"    Max absolute difference : {max_diff:.6e}")
#     print(f"    Weighted mean (precomp) : {mean_precomp*100:.4f}%")
#     print(f"    Weighted mean (rederived): {mean_rederived*100:.4f}%")
#     if not close:
#         n_diff = np.sum(~np.isclose(precomputed, rederived, equal_nan=True))
#         print(f"    Number of differing elements: {n_diff} / {len(precomputed)}")

# print("\n--- 100M ---")
# for name, (pre, red) in regions_to_check.items():
#     check_region(name, pre, red, label="100M")

# print("\n--- SIM ---")
# for name, (pre, red) in regions_to_check_sim.items():
#     check_region(name, pre, red, label="SIM")

# print("\n" + "=" * 60)
# print("WEIGHTS LENGTH CHECK")
# print("=" * 60)
# for name, (pre, _) in regions_to_check.items():
#     match = len(pre) == len(weights_new)
#     print(f"  {name}: len={len(pre)}, weights len={len(weights_new)}, match={match}")

In [ ]:
# polys = '/bp1/geog-tropical/data/BREATHE/shapefiles/BR_UF_2022.shp'
# gdf_states = gpd.read_file(polys)

# # Mapping of UF codes to macroregions
# macroregion_map = {
#     'RO': 'North', 'AC': 'North', 'AM': 'North', 'RR': 'North', 'PA': 'North', 'AP': 'North', 'TO': 'North',
#     'MA': 'Northeast', 'PI': 'Northeast', 'CE': 'Northeast', 'RN': 'Northeast', 'PB': 'Northeast',
#     'PE': 'Northeast', 'AL': 'Northeast', 'SE': 'Northeast', 'BA': 'Northeast',
#     'MT': 'Central-West', 'MS': 'Central-West', 'GO': 'Central-West', 'DF': 'Central-West',
#     'SP': 'Southeast', 'RJ': 'Southeast', 'MG': 'Southeast', 'ES': 'Southeast',
#     'PR': 'South', 'SC': 'South', 'RS': 'South'
# }

# gdf_states["macroregion"] = gdf_states["SIGLA_UF"].map(macroregion_map)
# macro = gdf_states.dissolve(by="macroregion")

# capitals = [
#     "Rio Branco", "Maceió", "Macapá", "Manaus", "Salvador", "Fortaleza", "Brasília",
#     "Vitória", "Goiânia", "São Luís", "Cuiabá", "Campo Grande", "Belo Horizonte",
#     "Belém", "João Pessoa", "Curitiba", "Recife", "Teresina", "Rio de Janeiro",
#     "Natal", "Porto Alegre", "Porto Velho", "Boa Vista", "Florianópolis",
#     "São Paulo", "Aracaju", "Palmas"
# ]

# regions = [
#     "North", "Northeast", "North", "North", "Northeast", "Northeast", "Central-West",
#     "Southeast", "Central-West", "Northeast", "Central-West", "Central-West", "Southeast",
#     "North", "Northeast", "South", "Northeast", "Northeast", "Southeast",
#     "Northeast", "South", "North", "North", "South",
#     "Southeast", "Northeast", "North"
# ]

In [ ]:
# df_merged = gdf.merge(mortality_estimates, on="rgi")
# missing_ids = set(gdf['rgi']) - set(df_merged['rgi'])
# gdf_missing = gdf[gdf['rgi'].isin(missing_ids)]


# df_merged_sim = gdf.merge(mortality_estimates_sim, on="rgi")
# missing_ids_sim = set(gdf['rgi']) - set(df_merged_sim['rgi'])
# gdf_missing_sim = gdf[gdf['rgi'].isin(missing_ids_sim)]

In [ ]:

# pop = pd.read_parquet('/bp1/geog-tropical/data/BREATHE/igr_population.parquet')
# # merged["rgi"] = merged["rgi"].astype(str)
# pop["rgi"] = pop["rgi"].astype(str)
# df_merged_pop = df_merged.merge(pop, on="rgi", how="left")
# df_merged_pop['mortality_anom_per_100000'] = (df_merged_pop['mean_attributable_mortality'] / df_merged_pop['population_count']) * 100000
# df_merged_pop['mortality_anom_lower_per_100000'] = (df_merged_pop['lower_ci'] / df_merged_pop['population_count']) * 100000
# df_merged_pop['mortality_anom_upper_per_100000'] = (df_merged_pop['upper_ci'] / df_merged_pop['population_count']) * 100000
# # df_merged_pop['mortality_anom_lower']

# df_merged_pop_sim = df_merged_sim.merge(pop, on="rgi", how="left")
# df_merged_pop_sim['mortality_anom_per_100000'] = (df_merged_pop_sim['mean_attributable_mortality'] / df_merged_pop_sim['population_count']) * 100000
# df_merged_pop_sim['mortality_anom_lower_per_100000'] = (df_merged_pop_sim['lower_ci'] / df_merged_pop_sim['population_count']) * 100000
# df_merged_pop_sim['mortality_anom_upper_per_100000'] = (df_merged_pop_sim['upper_ci'] / df_merged_pop_sim['population_count']) * 100000

In [ ]:
# descriptives = pd.read_excel('/user/home/al18709/breathe/data_analysis/Total_deaths_mean_time_series.xlsx')
# descriptives.rename(columns={'COD_IGR': 'rgi'},inplace=True)
# descriptives['rgi'] = descriptives['rgi'].astype(str)
# df_merged_with_non_accidental_mortality = pd.merge(df_merged_pop, descriptives[['rgi', 'total_deaths']], on='rgi', how='left')

In [ ]:
# # Merge in non-accidental death totals
# descriptives = pd.read_excel('/user/home/al18709/breathe/data_analysis/Results_COHORT_noRH_tas_ACCESS-CM2_r1i1p1f1.xlsx')
# descriptives.rename(columns={'COD_RGI': 'rgi'}, inplace=True)
# descriptives['rgi'] = descriptives['rgi'].astype(str)
# df_merged_with_non_accidental_mortality = pd.merge(
#     df_merged_pop, descriptives[['rgi', 'total_obitos']], on='rgi', how='left'
# )

# descriptives_sim = pd.read_excel('/user/home/al18709/breathe/data_analysis/Results_SIM_noRH_tas_ACCESS-CM2_r1i1p1f1.xlsx')
# descriptives_sim.rename(columns={'COD_RGI': 'rgi'}, inplace=True)
# descriptives_sim['rgi'] = descriptives_sim['rgi'].astype(str)
# df_sim = pd.merge(
#     df_merged_pop_sim, descriptives_sim[['rgi', 'total_obitos']], on='rgi', how='left'
# )

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Per-IGR comparison: 100M vs SIM mean_hr_fraction
# merged_check = df_merged_with_non_accidental_mortality[['rgi', 'mean_hr_fraction', 'population_count']].copy()
# merged_check = merged_check.merge(
#     mortality_estimates_sim[['rgi', 'mean_hr_fraction']].rename(columns={'mean_hr_fraction': 'mean_hr_fraction_sim'}),
#     on='rgi'
# )
# merged_check['diff'] = (merged_check['mean_hr_fraction'] - merged_check['mean_hr_fraction_sim']) * 100

# # Merge in IGR names
# merged_check = merged_check.merge(gdf[['rgi', 'nome_rgi']], on='rgi', how='left')

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # Left: histogram of differences
# axes[0].hist(merged_check['diff'], bins=40, edgecolor='k', alpha=0.7)
# axes[0].axvline(0, color='red', lw=1.5, ls='--')
# axes[0].set_xlabel('100M − SIM attributable fraction (percentage points)')
# axes[0].set_ylabel('Number of IGRs')
# axes[0].set_title('Distribution of per-IGR differences')

# n_lower = (merged_check['diff'] < 0).sum()
# n_higher = (merged_check['diff'] > 0).sum()
# axes[0].text(0.02, 0.97, f'Higher in SIM: {n_lower} IGRs\nHigher in 100M: {n_higher} IGRs',
#              transform=axes[0].transAxes, va='top', fontsize=11)

# # Right: scatter of difference vs population (log scale)
# sc = axes[1].scatter(
#     merged_check['population_count'],
#     merged_check['diff'],
#     alpha=0.5, s=20,
#     c=merged_check['diff'], cmap='coolwarm', vmin=-20, vmax=20
# )
# axes[1].axhline(0, color='red', lw=1.5, ls='--')
# axes[1].set_xscale('log')
# axes[1].set_xlabel('IGR population (log scale)')
# axes[1].set_ylabel('100M − SIM attributable fraction (percentage points)')
# axes[1].set_title('Difference vs population size')
# plt.colorbar(sc, ax=axes[1], label='Difference (pp)')

# # Annotate IGR with highest 100M hr_fraction
# top_hr = merged_check.loc[merged_check['mean_hr_fraction'].idxmax()]
# # axes[1].annotate(
# #     top_hr['nome_rgi'],
# #     xy=(top_hr['population_count'], top_hr['diff']),
# #     xytext=(15, 5), textcoords='offset points',
# #     fontsize=9, color='black',
# #     arrowprops=dict(arrowstyle='->', color='black', lw=1)
# # # )
# # axes[1].scatter(top_hr['population_count'], top_hr['diff'],
# #                 s=60, color='black', zorder=5, edgecolors='black', linewidths=0.8)

# # Annotate IGR with max percentage point difference
# top_diff = merged_check.loc[merged_check['diff'].idxmax()]
# axes[1].annotate(
#     top_diff['nome_rgi'],
#     xy=(top_diff['population_count'], top_diff['diff']),
#     xytext=(15, -10), textcoords='offset points',
#     fontsize=9, color='red',
#     arrowprops=dict(arrowstyle='->', color='red', lw=1)
# )
# axes[1].scatter(top_diff['population_count'], top_diff['diff'],
#                 s=60, color='red', zorder=5, edgecolors='black', linewidths=0.8)

# plt.tight_layout()
# plt.savefig('igr_fraction_diff.png', dpi=150, bbox_inches='tight')
# plt.show()

# # Summary
# print(f"IGRs where SIM is higher:  {n_lower} ({n_lower/len(merged_check)*100:.1f}%)")
# print(f"IGRs where 100M is higher: {n_higher} ({n_higher/len(merged_check)*100:.1f}%)")
# print(f"\nPopulation-weighted mean diff: {np.average(merged_check['diff'], weights=merged_check['population_count']):.2f} pp")
# print(f"Unweighted mean diff:          {merged_check['diff'].mean():.2f} pp")
# print(f"\nHighest 100M hr_fraction: {top_hr['nome_rgi']}, {top_hr['mean_hr_fraction']*100:.1f}%, diff: {top_hr['diff']:.1f} pp")
# print(f"Highest diff IGR: {top_diff['nome_rgi']}, diff: {top_diff['diff']:.1f} pp, 100M fraction: {top_diff['mean_hr_fraction']*100:.1f}%")

In [ ]:


# # === Add region to merged dataframe ===

# # df_merged_with_non_accidental_mortality['deaths_normalised'] = ((df_merged_with_non_accidental_mortality['mean_attributable_mortality'] / (df_merged_with_non_accidental_mortality['total_deaths']))) * 100
# # df_merged_with_non_accidental_mortality['deaths_lower_normalised'] = (df_merged_with_non_accidental_mortality['lower_ci']/ (df_merged_with_non_accidental_mortality['total_deaths'])) * 100
# # df_merged_with_non_accidental_mortality['deaths_upper_normalised'] = (df_merged_with_non_accidental_mortality['upper_ci']/ (df_merged_with_non_accidental_mortality['total_deaths'])) * 100

# df_sim = df_merged_pop_sim


# # === Create figure and gridspec layout ===
# fig = plt.figure(figsize=(26, 18))
# gs = gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[3, 1], wspace=0)

# # ========================
# # === LEFT: Brazil map ===
# # ========================

# ax_map = fig.add_subplot(gs[0, 0])
# af_climate = df_merged_with_non_accidental_mortality['mean_hr_fraction'] * 100
# af_climate_sim = df_sim['mean_hr_fraction'] * 100
# # af_climate = (df_merged_with_non_accidental_mortality['mean_attributable_mortality']/df_merged_with_non_accidental_mortality['mean_factual_mortality'])*100
# # af_climate_sim = (df_sim['mean_attributable_mortality']/df_sim['mean_factual_mortality'])*100
# gdf_diff = df_merged_with_non_accidental_mortality.copy()  # must be a GeoDataFrame
# gdf_diff['sim_af_climate_change'] = af_climate_sim
# gdf_diff['af_climate_change'] = af_climate
# gdf_diff['sim_100m_diff'] = af_climate - af_climate_sim

# # plot map
# ax_map, fig = plot_brazil(ax_map, fig, gdf_diff, var='sim_100m_diff',change_levels=[-12,-8,-4,0,4,8,12])

# # plot missing igrs
# gdf_missing.plot(ax=ax_map,color='none',edgecolor="#d2a87d",hatch='////',linewidth=0.1)
# ax_map.set_aspect('auto')
# ax_map.text(0.1, 1.048, 'a.', transform=ax_map.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

# macro.plot(ax=ax_map,edgecolor="#ffffff",linewidth=1.5, facecolor="none")


# size = 14
# fontsize = 18
# # Acre
# ax_map.scatter(-67.809, -9.974, color='black', s=size)
# ax_map.annotate('Rio Branco', xy=(-67.4, -10.6), xycoords='data', fontsize=fontsize)

# # Alagoas
# ax_map.scatter(-35.735, -9.666, color='black', s=size)
# ax_map.annotate('Maceió', xy=(-35.3, -10.2), xycoords='data', fontsize=fontsize)

# # Amapá
# ax_map.scatter(-51.050, 0.034, color='black', s=size)
# ax_map.annotate('Macapá', xy=(-50.6, -0.6 + 0.6), xycoords='data', fontsize=fontsize)

# # Amazonas
# ax_map.scatter(-60.021, -3.122, color='black', s=size)
# ax_map.annotate('Manaus', xy=(-59.6, -3.9), xycoords='data', fontsize=fontsize)

# # Bahia
# ax_map.scatter(-38.505, -12.986, color='black', s=size)
# ax_map.annotate('Salvador', xy=(-38.1 -0.1, -13.7 +0.1), xycoords='data', fontsize=fontsize)

# # Ceará
# ax_map.scatter(-38.529, -3.724, color='black', s=size)
# ax_map.annotate('Fortaleza', xy=(-38.1 - 0.1, -3.3 - 0.1), xycoords='data', fontsize=fontsize)

# # Distrito Federal
# ax_map.scatter(-47.895, -15.807, color='black', marker='s')
# ax_map.annotate('Brasília', xy=(-47.4 - 2.5, -16.6 + 1.3), xycoords='data', fontsize=fontsize)

# # Espírito Santo
# ax_map.scatter(-40.308, -20.315, color='black', s=size)
# ax_map.annotate('Vitória', xy=(-39.9, -21.0), xycoords='data', fontsize=fontsize)

# # Goiás
# ax_map.scatter(-49.264, -16.686, color='black', s=size)
# ax_map.annotate('Goiânia', xy=(-48.9, -17.4), xycoords='data', fontsize=fontsize)

# # Maranhão
# ax_map.scatter(-44.302, -2.530, color='black', s=size)
# ax_map.annotate('São Luís', xy=(-43.8 - 0.2, -3.2 + 0.8), xycoords='data', fontsize=fontsize)

# # Mato Grosso
# ax_map.scatter(-56.098, -15.601, color='black', s=size)
# ax_map.annotate('Cuiabá', xy=(-55.6 - 0.1, -16.3 + 0.2), xycoords='data', fontsize=fontsize)

# # Mato Grosso do Sul
# ax_map.scatter(-54.620, -20.469, color='black', s=size)
# ax_map.annotate('Campo Grande', xy=(-54.2 - 5, -21.2-0.25), xycoords='data', fontsize=fontsize)

# # Minas Gerais
# ax_map.scatter(-43.938, -19.920, color='black', s=size)
# ax_map.annotate('Belo Horizonte', xy=(-43.5, -20.7 + 0.9), xycoords='data', fontsize=fontsize)

# # Pará
# ax_map.scatter(-48.503, -1.455, color='black', s=size)
# ax_map.annotate('Belém', xy=(-48.0, -2.1), xycoords='data', fontsize=fontsize)

# # Paraíba
# ax_map.scatter(-34.864, -7.121, color='black', s=size)
# ax_map.annotate('João Pessoa', xy=(-34.4, -7.7+ 0.35), xycoords='data', fontsize=fontsize)

# # Paraná
# ax_map.scatter(-49.265, -25.428, color='black', s=size)
# ax_map.annotate('Curitiba', xy=(-48.8, -26.2), xycoords='data', fontsize=fontsize)

# # Pernambuco
# ax_map.scatter(-34.883, -8.059, color='black', s=size)
# ax_map.annotate('Recife', xy=(-34.5, -8.7), xycoords='data', fontsize=fontsize)

# # Piauí
# ax_map.scatter(-42.803, -5.090, color='black', s=size)
# ax_map.annotate('Teresina', xy=(-42.4 - 3, -5.8), xycoords='data', fontsize=fontsize)

# # Rio de Janeiro
# ax_map.scatter(-43.189, -22.917, color='black', s=size)
# ax_map.annotate('Rio de Janeiro', xy=(-42.8, -23.7), xycoords='data', fontsize=fontsize)

# # Rio Grande do Norte
# ax_map.scatter(-35.209, -5.795, color='black', s=size)
# ax_map.annotate('Natal', xy=(-34.8, -6.4 + 0.4), xycoords='data', fontsize=fontsize)

# # Rio Grande do Sul
# ax_map.scatter(-51.211, -30.038, color='black', s=size)
# ax_map.annotate('Porto Alegre', xy=(-50.8, -30.8), xycoords='data', fontsize=fontsize)

# # Rondônia
# ax_map.scatter(-63.903, -8.761, color='black', s=size)
# ax_map.annotate('Porto Velho', xy=(-63.4 - 0.1, -9.5 + 0.1), xycoords='data', fontsize=fontsize)

# # Roraima
# ax_map.scatter(-60.673, 2.823, color='black', s=size)
# ax_map.annotate('Boa Vista', xy=(-60.2, 2.1 + 0.5), xycoords='data', fontsize=fontsize)

# # Santa Catarina
# ax_map.scatter(-48.548, -27.595, color='black', s=size)
# ax_map.annotate('Florianópolis', xy=(-48.1, -28.3), xycoords='data', fontsize=fontsize)

# # São Paulo
# ax_map.scatter(-46.638, -23.568, color='black', s=size)
# ax_map.annotate('São Paulo', xy=(-46.3, -24.6), xycoords='data', fontsize=fontsize)

# # Sergipe
# ax_map.scatter(-37.073, -10.947, color='black', s=size)
# ax_map.annotate('Aracaju', xy=(-36.7, -11.6), xycoords='data', fontsize=fontsize)

# # Tocantins
# ax_map.scatter(-48.333, -10.184, color='black', s=size)
# ax_map.annotate('Palmas', xy=(-47.9, -10.9), xycoords='data', fontsize=fontsize)

# # ======================================
# # === RIGHT: Horizontal CI bar chart ===
# # ======================================


# ax_plot = fig.add_subplot(gs[0, 1])


# region_color_map = {
#     'Brazil':       '#444444',
#     'Central-West': '#1f77b4',
#     'North':        '#ff7f0e',
#     'Northeast':    '#2ca02c',
#     'South':        '#d62728',
#     'Southeast':    '#9467bd',
# }

# # (label, mean_100m, lo_100m, hi_100m, mean_sim, lo_sim, hi_sim)
# # Bottom to top: Central-West → South → Southeast → Northeast → North → Brazil
# chart_data = [
    
#     ('South',        south_af_cc*100,                south_af_cc_25*100,               south_af_cc_975*100,
#                      south_af_cc_sim*100,            south_af_cc_25_sim*100,           south_af_cc_975_sim*100),
#     ('Southeast',    southeast_af_cc*100,            southeast_af_cc_25*100,           southeast_af_cc_975*100,
#                      southeast_af_cc_sim*100,        southeast_af_cc_25_sim*100,       southeast_af_cc_975_sim*100),
#     ('Central-West', centralwest_af_cc*100,         centralwest_af_cc_25*100,         centralwest_af_cc_975*100,
#                      centralwest_af_cc_sim*100,      centralwest_af_cc_25_sim*100,     centralwest_af_cc_975_sim*100),
#     ('Northeast',    northeast_af_cc*100,            northeast_af_cc_25*100,           northeast_af_cc_975*100,
#                      northeast_af_cc_sim*100,        northeast_af_cc_25_sim*100,       northeast_af_cc_975_sim*100),
#     ('North',        north_af_cc*100,                north_af_cc_25*100,               north_af_cc_975*100,
#                      north_af_cc_sim*100,            north_af_cc_25_sim*100,           north_af_cc_975_sim*100),
#     ('Brazil',       all_brazil_af_climate_change*100, all_brazil_af_climate_change_25*100, all_brazil_af_climate_change_975*100,
#                      all_brazil_af_climate_change_sim*100, all_brazil_af_climate_change_25_sim*100, all_brazil_af_climate_change_975_sim*100),
# ]



# offset = 0.15
# row_spacing = 1.2
# ytick_pos    = []
# ytick_labels = []

# for i, (label, m100, lo100, hi100, msim, losim, hisim) in enumerate(chart_data):
#     color = region_color_map[label]
#     y = i * row_spacing
#     y_100m = y + offset
#     y_sim  = y - offset

#     # 100M — solid line, filled dot
#     ax_plot.plot([lo100, hi100], [y_100m, y_100m], color=color, lw=2.5, solid_capstyle='round')
#     ax_plot.plot(m100, y_100m, 'o', color=color, ms=9, zorder=5)

#     # SIM — dashed line, open dot
#     ax_plot.plot([losim, hisim], [y_sim, y_sim], color=color, lw=2.5, ls='--', solid_capstyle='round', alpha=0.75)
#     ax_plot.plot(msim, y_sim, 'o', color=color, ms=9, mfc='white', mew=2.5, zorder=5)

#     ytick_pos.append(y)
#     ytick_labels.append(label)

# # Separator between regions (index 4) and Brazil (index 5)
# ax_plot.axhline(y=4 * row_spacing + row_spacing / 2, color='#aaaaaa', lw=1, ls=':')
# ax_plot.set_ylim(-1, 8)




# # === Styling ===
# ax_plot.set_yticks(ytick_pos)
# ax_plot.set_yticklabels(ytick_labels, fontsize=18)
# ax_plot.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x)}'))
# ax_plot.tick_params(axis='x', labelsize=16)
# ax_plot.set_xlabel("Attributable fraction (%)", fontsize=20, labelpad=10)
# # ax_plot.set_title("Fraction of heat-related deaths attributable\nto climate change (95% CI)", fontsize=22, pad=10)
# ax_plot.grid(axis='x', linestyle='--', alpha=0.3)
# ax_plot.spines['left'].set_visible(False)
# ax_plot.spines['top'].set_visible(False)
# ax_plot.spines['right'].set_visible(False)
# ax_plot.yaxis.set_ticks_position('none')
# ax_plot.tick_params(axis='y', length=0)
# ax_plot.set_aspect('auto')
# ax_plot.set_title("Nationwide attributable mortality \n as a proportion of heat-related mortality\nwith 95% confidence intervals", fontsize=22,pad=10)


# # Legend for regions
# ax_plot.set_aspect('auto')

# # === regions ===

# # Add one map to the figure as a legend-style inset
# ax_map = fig.add_axes([0.88, 0.625, 0.18, 0.39])  # [left, bottom, width, height] in figure coords
# ax_map.axis('off')

# # Legend: 100M vs SIM
# from matplotlib.lines import Line2D
# legend_elements = [
#     Line2D([0], [0], color='#444444', lw=2.5, marker='o', markersize=8, label='100M'),
#     Line2D([0], [0], color='#444444', lw=2.5, ls='--', marker='o', markersize=8,
#            markerfacecolor='white', markeredgewidth=2, label='SIM'),
# ]
# ax_plot.legend(handles=legend_elements, fontsize=16, loc='lower right', framealpha=0.7)

# ax_plot.text(-0.28, 1.048, 'b.', transform=ax_plot.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

# # === Inset region map (top-right corner) ===
# # === Inset region map (top-right corner) ===
# ax_inset = fig.add_axes([0.88, 0.6, 0.18, 0.39])
# for region in macro.index:
#     color = region_color_map.get(region, 'lightgrey')
#     macro.loc[[region]].plot(ax=ax_inset, color=color, edgecolor='white', alpha=0.4)
# ax_inset.axis('off')
# for region, row in macro.iterrows():
#     centroid = row.geometry.centroid
#     ax_inset.text(
#         centroid.x, centroid.y, region,
#         color=region_color_map.get(region, 'black'),
#         fontsize=14, weight='bold',
#         ha='center', va='center'
#     )

# # Final layout and save
# ax_plot.text(-0.28, 1.048, 'b.', transform=ax_plot.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

# plt.tight_layout()
# plt.savefig('mortality_attribution_prop.png', dpi=600, bbox_inches='tight')
# plt.show()

In [ ]:
# https://pubmed.ncbi.nlm.nih.gov/34922344/

# the 100m has fewer elderly people than sim
# figure 2 of that paper

In [ ]:


# # === Add region to merged dataframe ===

# # df_merged_with_non_accidental_mortality['deaths_normalised'] = ((df_merged_with_non_accidental_mortality['mean_attributable_mortality'] / (df_merged_with_non_accidental_mortality['total_deaths']))) * 100
# # df_merged_with_non_accidental_mortality['deaths_lower_normalised'] = (df_merged_with_non_accidental_mortality['lower_ci']/ (df_merged_with_non_accidental_mortality['total_deaths'])) * 100
# # df_merged_with_non_accidental_mortality['deaths_upper_normalised'] = (df_merged_with_non_accidental_mortality['upper_ci']/ (df_merged_with_non_accidental_mortality['total_deaths'])) * 100

# df_sim = df_merged_pop_sim


# # === Create figure and gridspec layout ===
# fig = plt.figure(figsize=(26, 18))
# gs = gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[3, 1], wspace=0)

# # ========================
# # === LEFT: Brazil map ===
# # ========================

# ax_map = fig.add_subplot(gs[0, 0])
# af_climate = df_merged_with_non_accidental_mortality['mean_hr_fraction'] * 100
# af_climate_sim = df_sim['mean_hr_fraction'] * 100
# # af_climate = (df_merged_with_non_accidental_mortality['mean_attributable_mortality']/df_merged_with_non_accidental_mortality['mean_factual_mortality'])*100
# # af_climate_sim = (df_sim['mean_attributable_mortality']/df_sim['mean_factual_mortality'])*100
# gdf_diff = df_merged_with_non_accidental_mortality.copy()  # must be a GeoDataFrame
# gdf_diff['sim_af_climate_change'] = af_climate_sim
# gdf_diff['af_climate_change'] = af_climate
# gdf_diff['sim_100m_diff'] = af_climate - af_climate_sim

# # plot map
# ax_map, fig = plot_brazil(ax_map, fig, gdf_diff, var='sim_af_climate_change',change_levels=[0,20,40,60,80,100])  

# # plot missing igrs
# gdf_missing.plot(ax=ax_map,color='none',edgecolor="#d2a87d",hatch='////',linewidth=0.1)
# ax_map.set_aspect('auto')
# ax_map.text(0.1, 1.048, 'a.', transform=ax_map.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

# macro.plot(ax=ax_map,edgecolor="#ffffff",linewidth=1.5, facecolor="none")


# size = 14
# fontsize = 18
# # Acre
# ax_map.scatter(-67.809, -9.974, color='black', s=size)
# ax_map.annotate('Rio Branco', xy=(-67.4, -10.6), xycoords='data', fontsize=fontsize)

# # Alagoas
# ax_map.scatter(-35.735, -9.666, color='black', s=size)
# ax_map.annotate('Maceió', xy=(-35.3, -10.2), xycoords='data', fontsize=fontsize)

# # Amapá
# ax_map.scatter(-51.050, 0.034, color='black', s=size)
# ax_map.annotate('Macapá', xy=(-50.6, -0.6 + 0.6), xycoords='data', fontsize=fontsize)

# # Amazonas
# ax_map.scatter(-60.021, -3.122, color='black', s=size)
# ax_map.annotate('Manaus', xy=(-59.6, -3.9), xycoords='data', fontsize=fontsize)

# # Bahia
# ax_map.scatter(-38.505, -12.986, color='black', s=size)
# ax_map.annotate('Salvador', xy=(-38.1 -0.1, -13.7 +0.1), xycoords='data', fontsize=fontsize)

# # Ceará
# ax_map.scatter(-38.529, -3.724, color='black', s=size)
# ax_map.annotate('Fortaleza', xy=(-38.1 - 0.1, -3.3 - 0.1), xycoords='data', fontsize=fontsize)

# # Distrito Federal
# ax_map.scatter(-47.895, -15.807, color='black', marker='s')
# ax_map.annotate('Brasília', xy=(-47.4 - 2.5, -16.6 + 1.3), xycoords='data', fontsize=fontsize)

# # Espírito Santo
# ax_map.scatter(-40.308, -20.315, color='black', s=size)
# ax_map.annotate('Vitória', xy=(-39.9, -21.0), xycoords='data', fontsize=fontsize)

# # Goiás
# ax_map.scatter(-49.264, -16.686, color='black', s=size)
# ax_map.annotate('Goiânia', xy=(-48.9, -17.4), xycoords='data', fontsize=fontsize)

# # Maranhão
# ax_map.scatter(-44.302, -2.530, color='black', s=size)
# ax_map.annotate('São Luís', xy=(-43.8 - 0.2, -3.2 + 0.8), xycoords='data', fontsize=fontsize)

# # Mato Grosso
# ax_map.scatter(-56.098, -15.601, color='black', s=size)
# ax_map.annotate('Cuiabá', xy=(-55.6 - 0.1, -16.3 + 0.2), xycoords='data', fontsize=fontsize)

# # Mato Grosso do Sul
# ax_map.scatter(-54.620, -20.469, color='black', s=size)
# ax_map.annotate('Campo Grande', xy=(-54.2 - 5, -21.2-0.25), xycoords='data', fontsize=fontsize)

# # Minas Gerais
# ax_map.scatter(-43.938, -19.920, color='black', s=size)
# ax_map.annotate('Belo Horizonte', xy=(-43.5, -20.7 + 0.9), xycoords='data', fontsize=fontsize)

# # Pará
# ax_map.scatter(-48.503, -1.455, color='black', s=size)
# ax_map.annotate('Belém', xy=(-48.0, -2.1), xycoords='data', fontsize=fontsize)

# # Paraíba
# ax_map.scatter(-34.864, -7.121, color='black', s=size)
# ax_map.annotate('João Pessoa', xy=(-34.4, -7.7+ 0.35), xycoords='data', fontsize=fontsize)

# # Paraná
# ax_map.scatter(-49.265, -25.428, color='black', s=size)
# ax_map.annotate('Curitiba', xy=(-48.8, -26.2), xycoords='data', fontsize=fontsize)

# # Pernambuco
# ax_map.scatter(-34.883, -8.059, color='black', s=size)
# ax_map.annotate('Recife', xy=(-34.5, -8.7), xycoords='data', fontsize=fontsize)

# # Piauí
# ax_map.scatter(-42.803, -5.090, color='black', s=size)
# ax_map.annotate('Teresina', xy=(-42.4 - 3, -5.8), xycoords='data', fontsize=fontsize)

# # Rio de Janeiro
# ax_map.scatter(-43.189, -22.917, color='black', s=size)
# ax_map.annotate('Rio de Janeiro', xy=(-42.8, -23.7), xycoords='data', fontsize=fontsize)

# # Rio Grande do Norte
# ax_map.scatter(-35.209, -5.795, color='black', s=size)
# ax_map.annotate('Natal', xy=(-34.8, -6.4 + 0.4), xycoords='data', fontsize=fontsize)

# # Rio Grande do Sul
# ax_map.scatter(-51.211, -30.038, color='black', s=size)
# ax_map.annotate('Porto Alegre', xy=(-50.8, -30.8), xycoords='data', fontsize=fontsize)

# # Rondônia
# ax_map.scatter(-63.903, -8.761, color='black', s=size)
# ax_map.annotate('Porto Velho', xy=(-63.4 - 0.1, -9.5 + 0.1), xycoords='data', fontsize=fontsize)

# # Roraima
# ax_map.scatter(-60.673, 2.823, color='black', s=size)
# ax_map.annotate('Boa Vista', xy=(-60.2, 2.1 + 0.5), xycoords='data', fontsize=fontsize)

# # Santa Catarina
# ax_map.scatter(-48.548, -27.595, color='black', s=size)
# ax_map.annotate('Florianópolis', xy=(-48.1, -28.3), xycoords='data', fontsize=fontsize)

# # São Paulo
# ax_map.scatter(-46.638, -23.568, color='black', s=size)
# ax_map.annotate('São Paulo', xy=(-46.3, -24.6), xycoords='data', fontsize=fontsize)

# # Sergipe
# ax_map.scatter(-37.073, -10.947, color='black', s=size)
# ax_map.annotate('Aracaju', xy=(-36.7, -11.6), xycoords='data', fontsize=fontsize)

# # Tocantins
# ax_map.scatter(-48.333, -10.184, color='black', s=size)
# ax_map.annotate('Palmas', xy=(-47.9, -10.9), xycoords='data', fontsize=fontsize)

# # ======================================
# # === RIGHT: Horizontal CI bar chart ===
# # ======================================


# ax_plot = fig.add_subplot(gs[0, 1])


# region_color_map = {
#     'Brazil':       '#444444',
#     'Central-West': '#1f77b4',
#     'North':        '#ff7f0e',
#     'Northeast':    '#2ca02c',
#     'South':        '#d62728',
#     'Southeast':    '#9467bd',
# }

# # (label, mean_100m, lo_100m, hi_100m, mean_sim, lo_sim, hi_sim)
# # Bottom to top: Central-West → South → Southeast → Northeast → North → Brazil
# chart_data = [
    
#     ('South',        south_af_cc*100,                south_af_cc_25*100,               south_af_cc_975*100,
#                      south_af_cc_sim*100,            south_af_cc_25_sim*100,           south_af_cc_975_sim*100),
#     ('Southeast',    southeast_af_cc*100,            southeast_af_cc_25*100,           southeast_af_cc_975*100,
#                      southeast_af_cc_sim*100,        southeast_af_cc_25_sim*100,       southeast_af_cc_975_sim*100),
#     ('Central-West', centralwest_af_cc*100,         centralwest_af_cc_25*100,         centralwest_af_cc_975*100,
#                      centralwest_af_cc_sim*100,      centralwest_af_cc_25_sim*100,     centralwest_af_cc_975_sim*100),
#     ('Northeast',    northeast_af_cc*100,            northeast_af_cc_25*100,           northeast_af_cc_975*100,
#                      northeast_af_cc_sim*100,        northeast_af_cc_25_sim*100,       northeast_af_cc_975_sim*100),
#     ('North',        north_af_cc*100,                north_af_cc_25*100,               north_af_cc_975*100,
#                      north_af_cc_sim*100,            north_af_cc_25_sim*100,           north_af_cc_975_sim*100),
#     ('Brazil',       all_brazil_af_climate_change*100, all_brazil_af_climate_change_25*100, all_brazil_af_climate_change_975*100,
#                      all_brazil_af_climate_change_sim*100, all_brazil_af_climate_change_25_sim*100, all_brazil_af_climate_change_975_sim*100),
# ]

# offset = 0.15
# row_spacing = 1.2
# ytick_pos    = []
# ytick_labels = []

# for i, (label, m100, lo100, hi100, msim, losim, hisim) in enumerate(chart_data):
#     color = region_color_map[label]
#     y = i * row_spacing
#     y_100m = y + offset
#     y_sim  = y - offset

#     # 100M — solid line, filled dot
#     ax_plot.plot([lo100, hi100], [y_100m, y_100m], color=color, lw=2.5, solid_capstyle='round')
#     ax_plot.plot(m100, y_100m, 'o', color=color, ms=9, zorder=5)

#     # SIM — dashed line, open dot
#     ax_plot.plot([losim, hisim], [y_sim, y_sim], color=color, lw=2.5, ls='--', solid_capstyle='round', alpha=0.75)
#     ax_plot.plot(msim, y_sim, 'o', color=color, ms=9, mfc='white', mew=2.5, zorder=5)

#     ytick_pos.append(y)
#     ytick_labels.append(label)

# # Separator between regions (index 4) and Brazil (index 5)
# ax_plot.axhline(y=4 * row_spacing + row_spacing / 2, color='#aaaaaa', lw=1, ls=':')
# ax_plot.set_ylim(-1, 8)




# # === Styling ===
# ax_plot.set_yticks(ytick_pos)
# ax_plot.set_yticklabels(ytick_labels, fontsize=18)
# ax_plot.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x)}'))
# ax_plot.tick_params(axis='x', labelsize=16)
# ax_plot.set_xlabel("Attributable fraction (%)", fontsize=20, labelpad=10)
# # ax_plot.set_title("Fraction of heat-related deaths attributable\nto climate change (95% CI)", fontsize=22, pad=10)
# ax_plot.grid(axis='x', linestyle='--', alpha=0.3)
# ax_plot.spines['left'].set_visible(False)
# ax_plot.spines['top'].set_visible(False)
# ax_plot.spines['right'].set_visible(False)
# ax_plot.yaxis.set_ticks_position('none')
# ax_plot.tick_params(axis='y', length=0)
# ax_plot.set_aspect('auto')
# ax_plot.set_title("Nationwide attributable mortality \n as a proportion of heat-related mortality\nwith 95% confidence intervals", fontsize=22,pad=10)


# # Legend for regions
# ax_plot.set_aspect('auto')

# # === regions ===

# # Add one map to the figure as a legend-style inset
# ax_map = fig.add_axes([0.88, 0.625, 0.18, 0.39])  # [left, bottom, width, height] in figure coords
# ax_map.axis('off')

# # Legend: 100M vs SIM
# from matplotlib.lines import Line2D
# legend_elements = [
#     Line2D([0], [0], color='#444444', lw=2.5, marker='o', markersize=8, label='100M'),
#     Line2D([0], [0], color='#444444', lw=2.5, ls='--', marker='o', markersize=8,
#            markerfacecolor='white', markeredgewidth=2, label='SIM'),
# ]
# ax_plot.legend(handles=legend_elements, fontsize=16, loc='lower right', framealpha=0.7)

# ax_plot.text(-0.28, 1.048, 'b.', transform=ax_plot.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

# # === Inset region map (top-right corner) ===
# # === Inset region map (top-right corner) ===
# ax_inset = fig.add_axes([0.88, 0.6, 0.18, 0.39])
# for region in macro.index:
#     color = region_color_map.get(region, 'lightgrey')
#     macro.loc[[region]].plot(ax=ax_inset, color=color, edgecolor='white', alpha=0.4)
# ax_inset.axis('off')
# for region, row in macro.iterrows():
#     centroid = row.geometry.centroid
#     ax_inset.text(
#         centroid.x, centroid.y, region,
#         color=region_color_map.get(region, 'black'),
#         fontsize=14, weight='bold',
#         ha='center', va='center'
#     )

# # Final layout and save
# ax_plot.text(-0.28, 1.048, 'b.', transform=ax_plot.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

# plt.tight_layout()
# plt.savefig('mortality_attribution_prop.png', dpi=600, bbox_inches='tight')
# plt.show()

In [ ]:


# # === Add region to merged dataframe ===

# # df_merged_with_non_accidental_mortality['deaths_normalised'] = ((df_merged_with_non_accidental_mortality['mean_attributable_mortality'] / (df_merged_with_non_accidental_mortality['total_deaths']))) * 100
# # df_merged_with_non_accidental_mortality['deaths_lower_normalised'] = (df_merged_with_non_accidental_mortality['lower_ci']/ (df_merged_with_non_accidental_mortality['total_deaths'])) * 100
# # df_merged_with_non_accidental_mortality['deaths_upper_normalised'] = (df_merged_with_non_accidental_mortality['upper_ci']/ (df_merged_with_non_accidental_mortality['total_deaths'])) * 100

# df_sim = df_merged_pop_sim


# # === Create figure and gridspec layout ===
# fig = plt.figure(figsize=(26, 18))
# gs = gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[3, 1], wspace=0)

# # ========================
# # === LEFT: Brazil map ===
# # ========================

# ax_map = fig.add_subplot(gs[0, 0])
# af_climate = df_merged_with_non_accidental_mortality['mean_hr_fraction'] * 100
# af_climate_sim = df_sim['mean_hr_fraction'] * 100
# # af_climate = (df_merged_with_non_accidental_mortality['mean_attributable_mortality']/df_merged_with_non_accidental_mortality['mean_factual_mortality'])*100
# # af_climate_sim = (df_sim['mean_attributable_mortality']/df_sim['mean_factual_mortality'])*100
# gdf_diff = df_merged_with_non_accidental_mortality.copy()  # must be a GeoDataFrame
# gdf_diff['sim_af_climate_change'] = af_climate_sim
# gdf_diff['af_climate_change'] = af_climate
# gdf_diff['sim_100m_diff'] = af_climate - af_climate_sim

# # plot map
# ax_map, fig = plot_brazil(ax_map, fig, gdf_diff, var='af_climate_change',change_levels=[0,20,40,60,80,100])  

# # plot missing igrs
# gdf_missing.plot(ax=ax_map,color='none',edgecolor="#d2a87d",hatch='////',linewidth=0.1)
# ax_map.set_aspect('auto')
# ax_map.text(0.1, 1.048, 'a.', transform=ax_map.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

# macro.plot(ax=ax_map,edgecolor="#ffffff",linewidth=1.5, facecolor="none")


# size = 14
# fontsize = 18
# # Acre
# ax_map.scatter(-67.809, -9.974, color='black', s=size)
# ax_map.annotate('Rio Branco', xy=(-67.4, -10.6), xycoords='data', fontsize=fontsize)

# # Alagoas
# ax_map.scatter(-35.735, -9.666, color='black', s=size)
# ax_map.annotate('Maceió', xy=(-35.3, -10.2), xycoords='data', fontsize=fontsize)

# # Amapá
# ax_map.scatter(-51.050, 0.034, color='black', s=size)
# ax_map.annotate('Macapá', xy=(-50.6, -0.6 + 0.6), xycoords='data', fontsize=fontsize)

# # Amazonas
# ax_map.scatter(-60.021, -3.122, color='black', s=size)
# ax_map.annotate('Manaus', xy=(-59.6, -3.9), xycoords='data', fontsize=fontsize)

# # Bahia
# ax_map.scatter(-38.505, -12.986, color='black', s=size)
# ax_map.annotate('Salvador', xy=(-38.1 -0.1, -13.7 +0.1), xycoords='data', fontsize=fontsize)

# # Ceará
# ax_map.scatter(-38.529, -3.724, color='black', s=size)
# ax_map.annotate('Fortaleza', xy=(-38.1 - 0.1, -3.3 - 0.1), xycoords='data', fontsize=fontsize)

# # Distrito Federal
# ax_map.scatter(-47.895, -15.807, color='black', marker='s')
# ax_map.annotate('Brasília', xy=(-47.4 - 2.5, -16.6 + 1.3), xycoords='data', fontsize=fontsize)

# # Espírito Santo
# ax_map.scatter(-40.308, -20.315, color='black', s=size)
# ax_map.annotate('Vitória', xy=(-39.9, -21.0), xycoords='data', fontsize=fontsize)

# # Goiás
# ax_map.scatter(-49.264, -16.686, color='black', s=size)
# ax_map.annotate('Goiânia', xy=(-48.9, -17.4), xycoords='data', fontsize=fontsize)

# # Maranhão
# ax_map.scatter(-44.302, -2.530, color='black', s=size)
# ax_map.annotate('São Luís', xy=(-43.8 - 0.2, -3.2 + 0.8), xycoords='data', fontsize=fontsize)

# # Mato Grosso
# ax_map.scatter(-56.098, -15.601, color='black', s=size)
# ax_map.annotate('Cuiabá', xy=(-55.6 - 0.1, -16.3 + 0.2), xycoords='data', fontsize=fontsize)

# # Mato Grosso do Sul
# ax_map.scatter(-54.620, -20.469, color='black', s=size)
# ax_map.annotate('Campo Grande', xy=(-54.2 - 5, -21.2-0.25), xycoords='data', fontsize=fontsize)

# # Minas Gerais
# ax_map.scatter(-43.938, -19.920, color='black', s=size)
# ax_map.annotate('Belo Horizonte', xy=(-43.5, -20.7 + 0.9), xycoords='data', fontsize=fontsize)

# # Pará
# ax_map.scatter(-48.503, -1.455, color='black', s=size)
# ax_map.annotate('Belém', xy=(-48.0, -2.1), xycoords='data', fontsize=fontsize)

# # Paraíba
# ax_map.scatter(-34.864, -7.121, color='black', s=size)
# ax_map.annotate('João Pessoa', xy=(-34.4, -7.7+ 0.35), xycoords='data', fontsize=fontsize)

# # Paraná
# ax_map.scatter(-49.265, -25.428, color='black', s=size)
# ax_map.annotate('Curitiba', xy=(-48.8, -26.2), xycoords='data', fontsize=fontsize)

# # Pernambuco
# ax_map.scatter(-34.883, -8.059, color='black', s=size)
# ax_map.annotate('Recife', xy=(-34.5, -8.7), xycoords='data', fontsize=fontsize)

# # Piauí
# ax_map.scatter(-42.803, -5.090, color='black', s=size)
# ax_map.annotate('Teresina', xy=(-42.4 - 3, -5.8), xycoords='data', fontsize=fontsize)

# # Rio de Janeiro
# ax_map.scatter(-43.189, -22.917, color='black', s=size)
# ax_map.annotate('Rio de Janeiro', xy=(-42.8, -23.7), xycoords='data', fontsize=fontsize)

# # Rio Grande do Norte
# ax_map.scatter(-35.209, -5.795, color='black', s=size)
# ax_map.annotate('Natal', xy=(-34.8, -6.4 + 0.4), xycoords='data', fontsize=fontsize)

# # Rio Grande do Sul
# ax_map.scatter(-51.211, -30.038, color='black', s=size)
# ax_map.annotate('Porto Alegre', xy=(-50.8, -30.8), xycoords='data', fontsize=fontsize)

# # Rondônia
# ax_map.scatter(-63.903, -8.761, color='black', s=size)
# ax_map.annotate('Porto Velho', xy=(-63.4 - 0.1, -9.5 + 0.1), xycoords='data', fontsize=fontsize)

# # Roraima
# ax_map.scatter(-60.673, 2.823, color='black', s=size)
# ax_map.annotate('Boa Vista', xy=(-60.2, 2.1 + 0.5), xycoords='data', fontsize=fontsize)

# # Santa Catarina
# ax_map.scatter(-48.548, -27.595, color='black', s=size)
# ax_map.annotate('Florianópolis', xy=(-48.1, -28.3), xycoords='data', fontsize=fontsize)

# # São Paulo
# ax_map.scatter(-46.638, -23.568, color='black', s=size)
# ax_map.annotate('São Paulo', xy=(-46.3, -24.6), xycoords='data', fontsize=fontsize)

# # Sergipe
# ax_map.scatter(-37.073, -10.947, color='black', s=size)
# ax_map.annotate('Aracaju', xy=(-36.7, -11.6), xycoords='data', fontsize=fontsize)

# # Tocantins
# ax_map.scatter(-48.333, -10.184, color='black', s=size)
# ax_map.annotate('Palmas', xy=(-47.9, -10.9), xycoords='data', fontsize=fontsize)

# # ======================================
# # === RIGHT: Horizontal CI bar chart ===
# # ======================================


# ax_plot = fig.add_subplot(gs[0, 1])


# region_color_map = {
#     'Brazil':       '#444444',
#     'Central-West': '#1f77b4',
#     'North':        '#ff7f0e',
#     'Northeast':    '#2ca02c',
#     'South':        '#d62728',
#     'Southeast':    '#9467bd',
# }

# # (label, mean_100m, lo_100m, hi_100m, mean_sim, lo_sim, hi_sim)
# # Bottom to top: Central-West → South → Southeast → Northeast → North → Brazil
# chart_data = [
    
#     ('South',        south_af_cc*100,                south_af_cc_25*100,               south_af_cc_975*100,
#                      south_af_cc_sim*100,            south_af_cc_25_sim*100,           south_af_cc_975_sim*100),
#     ('Southeast',    southeast_af_cc*100,            southeast_af_cc_25*100,           southeast_af_cc_975*100,
#                      southeast_af_cc_sim*100,        southeast_af_cc_25_sim*100,       southeast_af_cc_975_sim*100),
#     ('Central-West', centralwest_af_cc*100,         centralwest_af_cc_25*100,         centralwest_af_cc_975*100,
#                      centralwest_af_cc_sim*100,      centralwest_af_cc_25_sim*100,     centralwest_af_cc_975_sim*100),
#     ('Northeast',    northeast_af_cc*100,            northeast_af_cc_25*100,           northeast_af_cc_975*100,
#                      northeast_af_cc_sim*100,        northeast_af_cc_25_sim*100,       northeast_af_cc_975_sim*100),
#     ('North',        north_af_cc*100,                north_af_cc_25*100,               north_af_cc_975*100,
#                      north_af_cc_sim*100,            north_af_cc_25_sim*100,           north_af_cc_975_sim*100),
#     ('Brazil',       all_brazil_af_climate_change*100, all_brazil_af_climate_change_25*100, all_brazil_af_climate_change_975*100,
#                      all_brazil_af_climate_change_sim*100, all_brazil_af_climate_change_25_sim*100, all_brazil_af_climate_change_975_sim*100),
# ]

# offset = 0.15
# row_spacing = 1.2
# ytick_pos    = []
# ytick_labels = []

# for i, (label, m100, lo100, hi100, msim, losim, hisim) in enumerate(chart_data):
#     color = region_color_map[label]
#     y = i * row_spacing
#     y_100m = y + offset
#     y_sim  = y - offset

#     # 100M — solid line, filled dot
#     ax_plot.plot([lo100, hi100], [y_100m, y_100m], color=color, lw=2.5, solid_capstyle='round')
#     ax_plot.plot(m100, y_100m, 'o', color=color, ms=9, zorder=5)

#     # SIM — dashed line, open dot
#     ax_plot.plot([losim, hisim], [y_sim, y_sim], color=color, lw=2.5, ls='--', solid_capstyle='round', alpha=0.75)
#     ax_plot.plot(msim, y_sim, 'o', color=color, ms=9, mfc='white', mew=2.5, zorder=5)

#     ytick_pos.append(y)
#     ytick_labels.append(label)

# # Separator between regions (index 4) and Brazil (index 5)
# ax_plot.axhline(y=4 * row_spacing + row_spacing / 2, color='#aaaaaa', lw=1, ls=':')
# ax_plot.set_ylim(-1, 8)




# # === Styling ===
# ax_plot.set_yticks(ytick_pos)
# ax_plot.set_yticklabels(ytick_labels, fontsize=18)
# ax_plot.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x)}'))
# ax_plot.tick_params(axis='x', labelsize=16)
# ax_plot.set_xlabel("Attributable fraction (%)", fontsize=20, labelpad=10)
# # ax_plot.set_title("Fraction of heat-related deaths attributable\nto climate change (95% CI)", fontsize=22, pad=10)
# ax_plot.grid(axis='x', linestyle='--', alpha=0.3)
# ax_plot.spines['left'].set_visible(False)
# ax_plot.spines['top'].set_visible(False)
# ax_plot.spines['right'].set_visible(False)
# ax_plot.yaxis.set_ticks_position('none')
# ax_plot.tick_params(axis='y', length=0)
# ax_plot.set_aspect('auto')
# ax_plot.set_title("Nationwide attributable mortality \n as a proportion of heat-related mortality\nwith 95% confidence intervals", fontsize=22,pad=10)


# # Legend for regions
# ax_plot.set_aspect('auto')

# # === regions ===

# # Add one map to the figure as a legend-style inset
# ax_map = fig.add_axes([0.88, 0.625, 0.18, 0.39])  # [left, bottom, width, height] in figure coords
# ax_map.axis('off')

# # Legend: 100M vs SIM
# from matplotlib.lines import Line2D
# legend_elements = [
#     Line2D([0], [0], color='#444444', lw=2.5, marker='o', markersize=8, label='100M'),
#     Line2D([0], [0], color='#444444', lw=2.5, ls='--', marker='o', markersize=8,
#            markerfacecolor='white', markeredgewidth=2, label='SIM'),
# ]
# ax_plot.legend(handles=legend_elements, fontsize=16, loc='lower right', framealpha=0.7)

# ax_plot.text(-0.28, 1.048, 'b.', transform=ax_plot.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

# # === Inset region map (top-right corner) ===
# # === Inset region map (top-right corner) ===
# ax_inset = fig.add_axes([0.88, 0.6, 0.18, 0.39])
# for region in macro.index:
#     color = region_color_map.get(region, 'lightgrey')
#     macro.loc[[region]].plot(ax=ax_inset, color=color, edgecolor='white', alpha=0.4)
# ax_inset.axis('off')
# for region, row in macro.iterrows():
#     centroid = row.geometry.centroid
#     ax_inset.text(
#         centroid.x, centroid.y, region,
#         color=region_color_map.get(region, 'black'),
#         fontsize=14, weight='bold',
#         ha='center', va='center'
#     )

# # Final layout and save
# ax_plot.text(-0.28, 1.048, 'b.', transform=ax_plot.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

# plt.tight_layout()
# plt.savefig('mortality_attribution_prop.png', dpi=600, bbox_inches='tight')
# plt.show()

In [ ]:


# # === Add region to merged dataframe ===

# # df_merged_with_non_accidental_mortality['deaths_normalised'] = ((df_merged_with_non_accidental_mortality['mean_attributable_mortality'] / (df_merged_with_non_accidental_mortality['total_deaths']))) * 100
# # df_merged_with_non_accidental_mortality['deaths_lower_normalised'] = (df_merged_with_non_accidental_mortality['lower_ci']/ (df_merged_with_non_accidental_mortality['total_deaths'])) * 100
# # df_merged_with_non_accidental_mortality['deaths_upper_normalised'] = (df_merged_with_non_accidental_mortality['upper_ci']/ (df_merged_with_non_accidental_mortality['total_deaths'])) * 100

# # df_sim = df_merged_pop_sim


# # === Create figure and gridspec layout ===
# fig = plt.figure(figsize=(26, 18))
# gs = gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[3, 1], wspace=0)

# # ========================
# # === LEFT: Brazil map ===
# # ========================

# ax_map = fig.add_subplot(gs[0, 0])
# # af_climate = df_merged_with_non_accidental_mortality['mean_hr_fraction'] * 100
# # af_climate_sim = df_sim['mean_hr_fraction'] * 100
# # # af_climate = (df_merged_with_non_accidental_mortality['mean_attributable_mortality']/df_merged_with_non_accidental_mortality['mean_factual_mortality'])*100
# # # af_climate_sim = (df_sim['mean_attributable_mortality']/df_sim['mean_factual_mortality'])*100
# gdf_diff = df_merged_with_non_accidental_mortality.copy()  # must be a GeoDataFrame
# # gdf_diff['sim_af_climate_change'] = af_climate_sim
# # gdf_diff['af_climate_change'] = af_climate
# # gdf_diff['sim_100m_diff'] = af_climate - af_climate_sim

# af = (df_merged_with_non_accidental_mortality['mean_attributable_mortality'] / df_merged_with_non_accidental_mortality['total_obitos']) * 100
# af_sim = (df_sim['mean_attributable_mortality'] / df_sim['total_obitos']) * 100
# gdf_diff['af'] = af
# gdf_diff['af_sim'] = af_sim
# gdf_diff['af_diff'] = af - af_sim

# # plot map
# ax_map, fig = plot_brazil(ax_map, fig, gdf_diff, var='af_diff',change_levels=[-2,-1.5,-1,-0.5,0,0.5,1,1.5,2])

# # plot missing igrs
# gdf_missing.plot(ax=ax_map,color='none',edgecolor="#d2a87d",hatch='////',linewidth=0.1)
# ax_map.set_aspect('auto')
# ax_map.text(0.1, 1.048, 'a.', transform=ax_map.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

# macro.plot(ax=ax_map,edgecolor="#ffffff",linewidth=1.5, facecolor="none")


# size = 14
# fontsize = 18
# # Acre
# ax_map.scatter(-67.809, -9.974, color='black', s=size)
# ax_map.annotate('Rio Branco', xy=(-67.4, -10.6), xycoords='data', fontsize=fontsize)

# # Alagoas
# ax_map.scatter(-35.735, -9.666, color='black', s=size)
# ax_map.annotate('Maceió', xy=(-35.3, -10.2), xycoords='data', fontsize=fontsize)

# # Amapá
# ax_map.scatter(-51.050, 0.034, color='black', s=size)
# ax_map.annotate('Macapá', xy=(-50.6, -0.6 + 0.6), xycoords='data', fontsize=fontsize)

# # Amazonas
# ax_map.scatter(-60.021, -3.122, color='black', s=size)
# ax_map.annotate('Manaus', xy=(-59.6, -3.9), xycoords='data', fontsize=fontsize)

# # Bahia
# ax_map.scatter(-38.505, -12.986, color='black', s=size)
# ax_map.annotate('Salvador', xy=(-38.1 -0.1, -13.7 +0.1), xycoords='data', fontsize=fontsize)

# # Ceará
# ax_map.scatter(-38.529, -3.724, color='black', s=size)
# ax_map.annotate('Fortaleza', xy=(-38.1 - 0.1, -3.3 - 0.1), xycoords='data', fontsize=fontsize)

# # Distrito Federal
# ax_map.scatter(-47.895, -15.807, color='black', marker='s')
# ax_map.annotate('Brasília', xy=(-47.4 - 2.5, -16.6 + 1.3), xycoords='data', fontsize=fontsize)

# # Espírito Santo
# ax_map.scatter(-40.308, -20.315, color='black', s=size)
# ax_map.annotate('Vitória', xy=(-39.9, -21.0), xycoords='data', fontsize=fontsize)

# # Goiás
# ax_map.scatter(-49.264, -16.686, color='black', s=size)
# ax_map.annotate('Goiânia', xy=(-48.9, -17.4), xycoords='data', fontsize=fontsize)

# # Maranhão
# ax_map.scatter(-44.302, -2.530, color='black', s=size)
# ax_map.annotate('São Luís', xy=(-43.8 - 0.2, -3.2 + 0.8), xycoords='data', fontsize=fontsize)

# # Mato Grosso
# ax_map.scatter(-56.098, -15.601, color='black', s=size)
# ax_map.annotate('Cuiabá', xy=(-55.6 - 0.1, -16.3 + 0.2), xycoords='data', fontsize=fontsize)

# # Mato Grosso do Sul
# ax_map.scatter(-54.620, -20.469, color='black', s=size)
# ax_map.annotate('Campo Grande', xy=(-54.2 - 5, -21.2-0.25), xycoords='data', fontsize=fontsize)

# # Minas Gerais
# ax_map.scatter(-43.938, -19.920, color='black', s=size)
# ax_map.annotate('Belo Horizonte', xy=(-43.5, -20.7 + 0.9), xycoords='data', fontsize=fontsize)

# # Pará
# ax_map.scatter(-48.503, -1.455, color='black', s=size)
# ax_map.annotate('Belém', xy=(-48.0, -2.1), xycoords='data', fontsize=fontsize)

# # Paraíba
# ax_map.scatter(-34.864, -7.121, color='black', s=size)
# ax_map.annotate('João Pessoa', xy=(-34.4, -7.7+ 0.35), xycoords='data', fontsize=fontsize)

# # Paraná
# ax_map.scatter(-49.265, -25.428, color='black', s=size)
# ax_map.annotate('Curitiba', xy=(-48.8, -26.2), xycoords='data', fontsize=fontsize)

# # Pernambuco
# ax_map.scatter(-34.883, -8.059, color='black', s=size)
# ax_map.annotate('Recife', xy=(-34.5, -8.7), xycoords='data', fontsize=fontsize)

# # Piauí
# ax_map.scatter(-42.803, -5.090, color='black', s=size)
# ax_map.annotate('Teresina', xy=(-42.4 - 3, -5.8), xycoords='data', fontsize=fontsize)

# # Rio de Janeiro
# ax_map.scatter(-43.189, -22.917, color='black', s=size)
# ax_map.annotate('Rio de Janeiro', xy=(-42.8, -23.7), xycoords='data', fontsize=fontsize)

# # Rio Grande do Norte
# ax_map.scatter(-35.209, -5.795, color='black', s=size)
# ax_map.annotate('Natal', xy=(-34.8, -6.4 + 0.4), xycoords='data', fontsize=fontsize)

# # Rio Grande do Sul
# ax_map.scatter(-51.211, -30.038, color='black', s=size)
# ax_map.annotate('Porto Alegre', xy=(-50.8, -30.8), xycoords='data', fontsize=fontsize)

# # Rondônia
# ax_map.scatter(-63.903, -8.761, color='black', s=size)
# ax_map.annotate('Porto Velho', xy=(-63.4 - 0.1, -9.5 + 0.1), xycoords='data', fontsize=fontsize)

# # Roraima
# ax_map.scatter(-60.673, 2.823, color='black', s=size)
# ax_map.annotate('Boa Vista', xy=(-60.2, 2.1 + 0.5), xycoords='data', fontsize=fontsize)

# # Santa Catarina
# ax_map.scatter(-48.548, -27.595, color='black', s=size)
# ax_map.annotate('Florianópolis', xy=(-48.1, -28.3), xycoords='data', fontsize=fontsize)

# # São Paulo
# ax_map.scatter(-46.638, -23.568, color='black', s=size)
# ax_map.annotate('São Paulo', xy=(-46.3, -24.6), xycoords='data', fontsize=fontsize)

# # Sergipe
# ax_map.scatter(-37.073, -10.947, color='black', s=size)
# ax_map.annotate('Aracaju', xy=(-36.7, -11.6), xycoords='data', fontsize=fontsize)

# # Tocantins
# ax_map.scatter(-48.333, -10.184, color='black', s=size)
# ax_map.annotate('Palmas', xy=(-47.9, -10.9), xycoords='data', fontsize=fontsize)

# # ======================================
# # === RIGHT: Horizontal CI bar chart ===
# # ======================================


# ax_plot = fig.add_subplot(gs[0, 1])


# region_color_map = {
#     'Brazil':       '#444444',
#     'Central-West': '#1f77b4',
#     'North':        '#ff7f0e',
#     'Northeast':    '#2ca02c',
#     'South':        '#d62728',
#     'Southeast':    '#9467bd',
# }

# # (label, mean_100m, lo_100m, hi_100m, mean_sim, lo_sim, hi_sim)
# # Bottom to top: Central-West → South → Southeast → Northeast → North → Brazil
# chart_data = [
    
#     ('South',        south_af_cc*100,                south_af_cc_25*100,               south_af_cc_975*100,
#                      south_af_cc_sim*100,            south_af_cc_25_sim*100,           south_af_cc_975_sim*100),
#     ('Southeast',    southeast_af_cc*100,            southeast_af_cc_25*100,           southeast_af_cc_975*100,
#                      southeast_af_cc_sim*100,        southeast_af_cc_25_sim*100,       southeast_af_cc_975_sim*100),
#     ('Central-West', centralwest_af_cc*100,         centralwest_af_cc_25*100,         centralwest_af_cc_975*100,
#                      centralwest_af_cc_sim*100,      centralwest_af_cc_25_sim*100,     centralwest_af_cc_975_sim*100),
#     ('Northeast',    northeast_af_cc*100,            northeast_af_cc_25*100,           northeast_af_cc_975*100,
#                      northeast_af_cc_sim*100,        northeast_af_cc_25_sim*100,       northeast_af_cc_975_sim*100),
#     ('North',        north_af_cc*100,                north_af_cc_25*100,               north_af_cc_975*100,
#                      north_af_cc_sim*100,            north_af_cc_25_sim*100,           north_af_cc_975_sim*100),
#     ('Brazil',       all_brazil_af_climate_change*100, all_brazil_af_climate_change_25*100, all_brazil_af_climate_change_975*100,
#                      all_brazil_af_climate_change_sim*100, all_brazil_af_climate_change_25_sim*100, all_brazil_af_climate_change_975_sim*100),
# ]

# offset = 0.15
# row_spacing = 1.2
# ytick_pos    = []
# ytick_labels = []

# for i, (label, m100, lo100, hi100, msim, losim, hisim) in enumerate(chart_data):
#     color = region_color_map[label]
#     y = i * row_spacing
#     y_100m = y + offset
#     y_sim  = y - offset

#     # 100M — solid line, filled dot
#     ax_plot.plot([lo100, hi100], [y_100m, y_100m], color=color, lw=2.5, solid_capstyle='round')
#     ax_plot.plot(m100, y_100m, 'o', color=color, ms=9, zorder=5)

#     # SIM — dashed line, open dot
#     ax_plot.plot([losim, hisim], [y_sim, y_sim], color=color, lw=2.5, ls='--', solid_capstyle='round', alpha=0.75)
#     ax_plot.plot(msim, y_sim, 'o', color=color, ms=9, mfc='white', mew=2.5, zorder=5)

#     ytick_pos.append(y)
#     ytick_labels.append(label)

# # Separator between regions (index 4) and Brazil (index 5)
# ax_plot.axhline(y=4 * row_spacing + row_spacing / 2, color='#aaaaaa', lw=1, ls=':')
# ax_plot.set_ylim(-1, 8)




# # === Styling ===
# ax_plot.set_yticks(ytick_pos)
# ax_plot.set_yticklabels(ytick_labels, fontsize=18)
# ax_plot.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x)}'))
# ax_plot.tick_params(axis='x', labelsize=16)
# ax_plot.set_xlabel("Attributable fraction (%)", fontsize=20, labelpad=10)
# # ax_plot.set_title("Fraction of heat-related deaths attributable\nto climate change (95% CI)", fontsize=22, pad=10)
# ax_plot.grid(axis='x', linestyle='--', alpha=0.3)
# ax_plot.spines['left'].set_visible(False)
# ax_plot.spines['top'].set_visible(False)
# ax_plot.spines['right'].set_visible(False)
# ax_plot.yaxis.set_ticks_position('none')
# ax_plot.tick_params(axis='y', length=0)
# ax_plot.set_aspect('auto')
# ax_plot.set_title("Nationwide attributable mortality \n as a proportion of heat-related mortality\nwith 95% confidence intervals", fontsize=22,pad=10)


# # Legend for regions
# ax_plot.set_aspect('auto')

# # === regions ===

# # Add one map to the figure as a legend-style inset
# ax_map = fig.add_axes([0.88, 0.625, 0.18, 0.39])  # [left, bottom, width, height] in figure coords
# ax_map.axis('off')

# # Legend: 100M vs SIM
# from matplotlib.lines import Line2D
# legend_elements = [
#     Line2D([0], [0], color='#444444', lw=2.5, marker='o', markersize=8, label='100M'),
#     Line2D([0], [0], color='#444444', lw=2.5, ls='--', marker='o', markersize=8,
#            markerfacecolor='white', markeredgewidth=2, label='SIM'),
# ]
# ax_plot.legend(handles=legend_elements, fontsize=16, loc='lower right', framealpha=0.7)

# ax_plot.text(-0.28, 1.048, 'b.', transform=ax_plot.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

# # === Inset region map (top-right corner) ===
# # === Inset region map (top-right corner) ===
# ax_inset = fig.add_axes([0.88, 0.6, 0.18, 0.39])
# for region in macro.index:
#     color = region_color_map.get(region, 'lightgrey')
#     macro.loc[[region]].plot(ax=ax_inset, color=color, edgecolor='white', alpha=0.4)
# ax_inset.axis('off')
# for region, row in macro.iterrows():
#     centroid = row.geometry.centroid
#     ax_inset.text(
#         centroid.x, centroid.y, region,
#         color=region_color_map.get(region, 'black'),
#         fontsize=14, weight='bold',
#         ha='center', va='center'
#     )

# # Final layout and save
# ax_plot.text(-0.28, 1.048, 'b.', transform=ax_plot.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')

# plt.tight_layout()
# plt.savefig('mortality_attribution_prop.png', dpi=600, bbox_inches='tight')
# plt.show()

In [ ]:
# # Find values above the 97.5th percentile
# outliers = af_cc_centralwest[af_cc_centralwest > 1]
# print("Number of outliers:", len(outliers))
# print("Outlier values:", outliers)

# # Optionally, see the largest values
# print("Top 10 largest values:")
# print(np.sort(af_cc_centralwest)[-10:])

In [ ]:
# print("Any negative values in centralwest_cfactuals?", np.any(centralwest_cfactuals < 0))
# print("Min value in centralwest_cfactuals:", np.min(centralwest_cfactuals))

In [ ]:
# neg_indices = np.where(centralwest_cfactuals < 0)[0]
# print("Problematic factuals:", centralwest_factuals[neg_indices])
# print("Problematic cfactuals:", centralwest_cfactuals[neg_indices])

In [ ]:
# import numpy as np
# print("Mean:", np.average(af_cc_centralwest, weights=weights_new))
# print("2.5th, 97.5th percentiles:", weighted_percentile(af_cc_centralwest, weights_new, [2.5, 97.5]))
# print("Any NaNs?", np.isnan(af_cc_centralwest).any())
# print("Any infs?", np.isinf(af_cc_centralwest).any())